In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/products.csv
/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/reviews.csv
/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/experiments.csv
/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/customers.csv
/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/recommendation_events.csv
/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/transactions.csv
/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/sessions.csv


In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

customers = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/customers.csv')
transactions = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/transactions.csv')
sessions = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/sessions.csv')
reviews = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/reviews.csv')
products = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/products.csv')
experiments = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/experiments.csv')
recommendation_events = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/recommendation_events.csv')


print("Customers    :", customers.shape)
print("Transactions :", transactions.shape)
print("Sessions     :", sessions.shape)
print("Reviews      :", reviews.shape)
print("Products     :", products.shape)
print("Experiments  :",experiments.shape)
print("Recommendation_event :",recommendation_events.shape)

Customers    : (200000, 13)
Transactions : (3000000, 12)
Sessions     : (2000000, 11)
Reviews      : (500000, 9)
Products     : (10000, 13)
Experiments  : (399996, 11)
Recommendation_event : (600000, 10)


In [6]:
import pandas as pd
import duckdb

con = duckdb.connect()

customers = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/customers.csv')
products = pd.read_csv('//kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/products.csv')
transactions = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/transactions.csv')
sessions = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/sessions.csv')
reviews = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/reviews.csv')
experiments = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/experiments.csv')
recommendation_events = pd.read_csv('/kaggle/input/datasets/piyushxx7/under-armour-ecommerce-behavior-dataset/recommendation_events.csv')

con.register('customers', customers)
con.register('products', products)
con.register('transactions', transactions)
con.register('sessions', sessions)
con.register('reviews', reviews)
con.register('experiments', experiments)
con.register('recommendation_events', recommendation_events)

print("All tables loaded and registered successfully")

All tables loaded and registered successfully


# **Row count validation**

In [7]:
query = """
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'transactions', COUNT(*) FROM transactions
UNION ALL
SELECT 'sessions', COUNT(*) FROM sessions
UNION ALL
SELECT 'reviews', COUNT(*) FROM reviews
UNION ALL
SELECT 'experiments', COUNT(*) FROM experiments
UNION ALL
SELECT 'recommendation_events', COUNT(*) FROM recommendation_events;
"""

con.execute(query).df()

,table_name,row_count
0,customers,200000
1,products,10000
2,transactions,3000000
3,sessions,2000000
4,reviews,500000
5,experiments,399996
6,recommendation_events,600000


# **Duplicate Primary Key Check**

In [8]:
query = """
SELECT 'customers' AS table_name, 
       COUNT(*) AS total_rows,
       COUNT(DISTINCT customer_id) AS unique_ids,
       COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_ids
FROM customers

UNION ALL

SELECT 'products',
       COUNT(*),
       COUNT(DISTINCT product_id),
       COUNT(*) - COUNT(DISTINCT product_id)
FROM products

UNION ALL

SELECT 'transactions',
       COUNT(*),
       COUNT(DISTINCT transaction_id),
       COUNT(*) - COUNT(DISTINCT transaction_id)
FROM transactions

UNION ALL

SELECT 'sessions',
       COUNT(*),
       COUNT(DISTINCT session_id),
       COUNT(*) - COUNT(DISTINCT session_id)
FROM sessions

UNION ALL

SELECT 'reviews',
       COUNT(*),
       COUNT(DISTINCT review_id),
       COUNT(*) - COUNT(DISTINCT review_id)
FROM reviews

UNION ALL

SELECT 'experiments',
       COUNT(*),
       COUNT(DISTINCT experiment_id || '-' || customer_id || '-' || COALESCE(session_id, 'NO_SESSION') || '-' || exposure_date),
       COUNT(*) - COUNT(DISTINCT experiment_id || '-' || customer_id || '-' || COALESCE(session_id, 'NO_SESSION') || '-' || exposure_date)
FROM experiments

UNION ALL

SELECT 'recommendation_events',
       COUNT(*),
       COUNT(DISTINCT event_id),
       COUNT(*) - COUNT(DISTINCT event_id)
FROM recommendation_events;
"""

con.execute(query).df()

,table_name,total_rows,unique_ids,duplicate_ids
0,customers,200000,200000,0
1,products,10000,10000,0
2,transactions,3000000,3000000,0
3,sessions,2000000,2000000,0
4,reviews,500000,500000,0
5,experiments,399996,399995,1
6,recommendation_events,600000,600000,0


# **Find duplicate experiment row**

In [9]:
query = """
SELECT
    experiment_id,
    customer_id,
    COALESCE(session_id, 'NO_SESSION') AS session_id_clean,
    exposure_date,
    COUNT(*) AS duplicate_count
FROM experiments
GROUP BY
    experiment_id,
    customer_id,
    COALESCE(session_id, 'NO_SESSION'),
    exposure_date
HAVING COUNT(*) > 1;
"""

con.execute(query).df()

,experiment_id,customer_id,session_id_clean,exposure_date,duplicate_count
0,EXP001,UA-C052033,NO_SESSION,2021-05-09,2


# **Create clean experiments view**

In [10]:
query = """
CREATE OR REPLACE VIEW experiments_clean AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY
                experiment_id,
                customer_id,
                COALESCE(session_id, 'NO_SESSION'),
                exposure_date
            ORDER BY exposure_date
        ) AS rn
    FROM experiments
)
WHERE rn = 1;
"""

con.execute(query)

print("experiments_clean view created")

experiments_clean view created


# **verify clean row count**

In [11]:
query = """
SELECT
    COUNT(*) AS clean_rows,
    COUNT(DISTINCT experiment_id || '-' || customer_id || '-' || COALESCE(session_id, 'NO_SESSION') || '-' || exposure_date) AS unique_experiment_rows,
    COUNT(*) - COUNT(DISTINCT experiment_id || '-' || customer_id || '-' || COALESCE(session_id, 'NO_SESSION') || '-' || exposure_date) AS duplicate_rows
FROM experiments_clean;
"""

con.execute(query).df()

,clean_rows,unique_experiment_rows,duplicate_rows
0,399995,399995,0


# **Check foreign key relationship integrity.**

In [12]:
query = """
SELECT 
    'transactions → customers' AS relationship,
    COUNT(*) AS missing_rows
FROM transactions t
LEFT JOIN customers c
    ON t.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 
    'transactions → products',
    COUNT(*)
FROM transactions t
LEFT JOIN products p
    ON t.product_id = p.product_id
WHERE p.product_id IS NULL

UNION ALL

SELECT 
    'sessions → customers',
    COUNT(*)
FROM sessions s
LEFT JOIN customers c
    ON s.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 
    'reviews → customers',
    COUNT(*)
FROM reviews r
LEFT JOIN customers c
    ON r.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 
    'reviews → products',
    COUNT(*)
FROM reviews r
LEFT JOIN products p
    ON r.product_id = p.product_id
WHERE p.product_id IS NULL

UNION ALL

SELECT 
    'experiments_clean → customers',
    COUNT(*)
FROM experiments_clean e
LEFT JOIN customers c
    ON e.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 
    'experiments_clean → sessions',
    COUNT(*)
FROM experiments_clean e
LEFT JOIN sessions s
    ON e.session_id = s.session_id
WHERE e.session_id IS NOT NULL
  AND s.session_id IS NULL

UNION ALL

SELECT 
    'recommendation_events → customers',
    COUNT(*)
FROM recommendation_events re
LEFT JOIN customers c
    ON re.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 
    'recommendation_events → products',
    COUNT(*)
FROM recommendation_events re
LEFT JOIN products p
    ON re.product_id = p.product_id
WHERE p.product_id IS NULL

UNION ALL

SELECT 
    'recommendation_events → sessions',
    COUNT(*)
FROM recommendation_events re
LEFT JOIN sessions s
    ON re.session_id = s.session_id
WHERE re.session_id IS NOT NULL
  AND s.session_id IS NULL;
"""

con.execute(query).df()

,relationship,missing_rows
0,transactions → customers,0
1,transactions → products,0
2,sessions → customers,0
3,reviews → customers,0
4,reviews → products,0
5,experiments_clean → customers,0
6,experiments_clean → sessions,0
7,recommendation_events → customers,0
8,recommendation_events → products,0
9,recommendation_events → sessions,0


# **Date Range Validation**

In [13]:
query = """
SELECT
    'customers.signup_date' AS date_field,
    MIN(CAST(signup_date AS DATE)) AS min_date,
    MAX(CAST(signup_date AS DATE)) AS max_date
FROM customers

UNION ALL

SELECT
    'transactions.transaction_date',
    MIN(CAST(transaction_date AS DATE)),
    MAX(CAST(transaction_date AS DATE))
FROM transactions

UNION ALL

SELECT
    'sessions.session_date',
    MIN(CAST(session_date AS DATE)),
    MAX(CAST(session_date AS DATE))
FROM sessions

UNION ALL

SELECT
    'reviews.review_date',
    MIN(CAST(review_date AS DATE)),
    MAX(CAST(review_date AS DATE))
FROM reviews

UNION ALL

SELECT
    'experiments_clean.exposure_date',
    MIN(CAST(exposure_date AS DATE)),
    MAX(CAST(exposure_date AS DATE))
FROM experiments_clean

UNION ALL

SELECT
    'recommendation_events.event_date',
    MIN(CAST(event_date AS DATE)),
    MAX(CAST(event_date AS DATE))
FROM recommendation_events;
"""

con.execute(query).df()

,date_field,min_date,max_date
0,customers.signup_date,2019-01-01,2025-12-30
1,transactions.transaction_date,2021-01-01,2026-02-27
2,sessions.session_date,2021-01-01,2026-02-27
3,reviews.review_date,2021-01-01,2026-02-27
4,experiments_clean.exposure_date,2021-03-01,2026-01-30
5,recommendation_events.event_date,2021-01-01,2026-02-27


# **Create Clean Views**

In [14]:
query = """
CREATE OR REPLACE VIEW customers_clean AS
SELECT
    customer_id,
    CAST(signup_date AS DATE) AS signup_date,
    age,
    COALESCE(gender, 'Unknown') AS gender,
    country,
    segment,
    is_churned,
    COALESCE(lifetime_value, 0) AS lifetime_value,
    is_loyalty_member,
    ua_rewards_points,
    email_opt_in,
    has_app,
    COALESCE(preferred_sport, 'Unknown') AS preferred_sport
FROM customers;

CREATE OR REPLACE VIEW products_clean AS
SELECT
    product_id,
    product_name,
    category,
    brand,
    gender_target,
    price,
    avg_rating,
    num_ratings,
    stock_quantity,
    discount_pct,
    is_featured,
    is_new_arrival,
    weight_kg
FROM products;

CREATE OR REPLACE VIEW transactions_clean AS
SELECT
    transaction_id,
    customer_id,
    product_id,
    CAST(transaction_date AS TIMESTAMP) AS transaction_date,
    quantity,
    unit_price,
    total_amount,
    discount_applied,
    loyalty_discount,
    status,
    payment_method,
    shipping_cost
FROM transactions;

CREATE OR REPLACE VIEW sessions_clean AS
SELECT
    session_id,
    customer_id,
    CAST(session_date AS TIMESTAMP) AS session_date,
    device,
    COALESCE(channel, 'unknown') AS channel,
    duration_seconds,
    pages_viewed,
    converted,
    bounced,
    cart_additions,
    is_loyalty_session
FROM sessions;

CREATE OR REPLACE VIEW reviews_clean AS
SELECT
    review_id,
    customer_id,
    product_id,
    CAST(review_date AS DATE) AS review_date,
    rating,
    review_text,
    helpful_votes,
    verified_purchase,
    review_source
FROM reviews;

CREATE OR REPLACE VIEW recommendation_events_clean AS
SELECT
    event_id,
    customer_id,
    session_id,
    product_id,
    recommendation_model,
    CAST(event_date AS TIMESTAMP) AS event_date,
    impression,
    clicked,
    purchased,
    COALESCE(revenue, 0) AS revenue
FROM recommendation_events;
"""

con.execute(query)

print("Clean views created successfully")

Clean views created successfully


In [15]:
query = """
SELECT 'customers_clean' AS table_name, COUNT(*) AS row_count FROM customers_clean
UNION ALL
SELECT 'products_clean', COUNT(*) FROM products_clean
UNION ALL
SELECT 'transactions_clean', COUNT(*) FROM transactions_clean
UNION ALL
SELECT 'sessions_clean', COUNT(*) FROM sessions_clean
UNION ALL
SELECT 'reviews_clean', COUNT(*) FROM reviews_clean
UNION ALL
SELECT 'experiments_clean', COUNT(*) FROM experiments_clean
UNION ALL
SELECT 'recommendation_events_clean', COUNT(*) FROM recommendation_events_clean;
"""

con.execute(query).df()

,table_name,row_count
0,customers_clean,200000
1,products_clean,10000
2,transactions_clean,3000000
3,sessions_clean,2000000
4,reviews_clean,500000
5,experiments_clean,399995
6,recommendation_events_clean,600000


# **Executive Customer KPI Overview**

In [16]:
query = """
SELECT
    COUNT(*) AS total_customers,

    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,

    ROUND(
        100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS retention_rate_pct,

    SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) AS loyalty_members,

    ROUND(
        100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS loyalty_member_pct,

    SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) AS app_users,

    ROUND(
        100.0 * SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS app_user_pct,

    SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) AS email_opt_in_users,

    ROUND(
        100.0 * SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS email_opt_in_pct,

    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value
FROM customers_clean;
"""

con.execute(query).df()

,total_customers,active_customers,churned_customers,churn_rate_pct,retention_rate_pct,loyalty_members,loyalty_member_pct,app_users,app_user_pct,email_opt_in_users,email_opt_in_pct,avg_lifetime_value
0,200000,162869.0,37131.0,18.57,81.43,109012.0,54.51,123712.0,61.86,143963.0,71.98,2219.95


**Which customer segments are causing the 18.57% churn?**
# **Churn Analysis by Customer Segment** 

In [17]:
query = """
SELECT
    segment,
    COUNT(*) AS total_customers,

    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,

    ROUND(
        100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_pct,

    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,

    SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) AS loyalty_members,

    ROUND(
        100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS loyalty_member_pct

FROM customers_clean
GROUP BY segment
ORDER BY churn_rate_pct DESC;
"""

con.execute(query).df()

,segment,total_customers,churned_customers,active_customers,churn_rate_pct,avg_lifetime_value,loyalty_members,loyalty_member_pct
0,One-Time Buyer,14099,7769.0,6330.0,55.10,656.17,4983.0,35.34
1,Discount Hunter,29872,9691.0,20181.0,32.44,781.11,10333.0,34.59
2,Casual Buyer,56057,12311.0,43746.0,21.96,1123.41,19608.0,34.98
3,UA Rewards Member,44240,3951.0,40289.0,8.93,2764.40,44240.0,100.00
4,Performance Athlete,39854,2773.0,37081.0,6.96,3460.63,13970.0,35.05
5,UA Rewards Elite,15878,636.0,15242.0,4.01,5555.71,15878.0,100.00


# **Churn by Loyalty Membership**

In [18]:
query = """
SELECT
    CASE 
        WHEN is_loyalty_member = 1 THEN 'Loyalty Member'
        ELSE 'Non-Loyalty Member'
    END AS loyalty_status,

    COUNT(*) AS total_customers,

    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,

    ROUND(
        100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_pct,

    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,

    ROUND(AVG(ua_rewards_points), 2) AS avg_rewards_points,

    ROUND(
        100.0 * SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS app_user_pct,

    ROUND(
        100.0 * SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS email_opt_in_pct

FROM customers_clean
GROUP BY loyalty_status
ORDER BY churn_rate_pct DESC;
"""

con.execute(query).df()

,loyalty_status,total_customers,churned_customers,active_customers,churn_rate_pct,avg_lifetime_value,avg_rewards_points,app_user_pct,email_opt_in_pct
0,Non-Loyalty Member,90988,21175.0,69813.0,23.27,1667.09,0.00,61.90,71.87
1,Loyalty Member,109012,15956.0,93056.0,14.64,2681.40,7547.54,61.82,72.08


# **Churn By App Usage**

In [19]:
query = """
SELECT
    CASE 
        WHEN has_app = 1 THEN 'App User'
        ELSE 'Non-App User'
    END AS app_status,

    COUNT(*) AS total_customers,

    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,

    ROUND(
        100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_pct,

    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,

    ROUND(
        100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS loyalty_member_pct,

    ROUND(
        100.0 * SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS email_opt_in_pct

FROM customers_clean
GROUP BY app_status
ORDER BY churn_rate_pct DESC;
"""

con.execute(query).df()

,app_status,total_customers,churned_customers,active_customers,churn_rate_pct,avg_lifetime_value,loyalty_member_pct,email_opt_in_pct
0,App User,123712,23210.0,100502.0,18.76,2213.11,54.47,71.99
1,Non-App User,76288,13921.0,62367.0,18.25,2231.04,54.56,71.97


# **Churn By Email Opt-in**

In [20]:
query = """
SELECT
    CASE 
        WHEN email_opt_in = 1 THEN 'Email Opt-in'
        ELSE 'No Email Opt-in'
    END AS email_status,

    COUNT(*) AS total_customers,

    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,

    ROUND(
        100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_pct,

    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,

    ROUND(
        100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS loyalty_member_pct,

    ROUND(
        100.0 * SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS app_user_pct

FROM customers_clean
GROUP BY email_status
ORDER BY churn_rate_pct DESC;
"""

con.execute(query).df()

,email_status,total_customers,churned_customers,active_customers,churn_rate_pct,avg_lifetime_value,loyalty_member_pct,app_user_pct
0,No Email Opt-in,56037,10500.0,45537.0,18.74,2219.97,54.32,61.84
1,Email Opt-in,143963,26631.0,117332.0,18.50,2219.95,54.58,61.86


# **Churn by Country**

In [21]:
query = """
SELECT
    country,
    COUNT(*) AS total_customers,

    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,

    ROUND(
        100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_pct,

    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,

    ROUND(
        100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS loyalty_member_pct,

    ROUND(
        100.0 * SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS app_user_pct,

    ROUND(
        100.0 * SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS email_opt_in_pct

FROM customers_clean
GROUP BY country
ORDER BY churn_rate_pct DESC;
"""

con.execute(query).df()

,country,total_customers,churned_customers,active_customers,churn_rate_pct,avg_lifetime_value,loyalty_member_pct,app_user_pct,email_opt_in_pct
0,KR,4118,784.0,3334.0,19.04,2238.98,54.81,61.07,71.20
1,JP,10010,1904.0,8106.0,19.02,2209.57,53.69,63.21,71.67
2,DE,15995,2981.0,13014.0,18.64,2215.51,54.85,61.90,72.29
3,CN,14032,2614.0,11418.0,18.63,2219.78,54.71,61.54,71.79
4,UK,24021,4468.0,19553.0,18.60,2221.68,54.72,61.52,72.07
5,US,89743,16667.0,73076.0,18.57,2224.40,54.54,61.96,71.91
6,CA,19986,3705.0,16281.0,18.54,2231.49,54.71,61.69,71.98
7,FR,7993,1463.0,6530.0,18.30,2181.37,53.52,61.60,72.13
8,AU,8086,1463.0,6623.0,18.09,2202.77,54.20,62.06,73.03
9,BR,6016,1082.0,4934.0,17.99,2199.12,54.01,61.14,71.84


# **Repeat Purchase Rate**

In [22]:
query = """
WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(DISTINCT transaction_id) AS total_orders,
        SUM(total_amount) AS total_revenue,
        MIN(CAST(transaction_date AS DATE)) AS first_purchase_date,
        MAX(CAST(transaction_date AS DATE)) AS last_purchase_date
    FROM transactions_clean
    WHERE status = 'completed'
    GROUP BY customer_id
)

SELECT
    COUNT(*) AS purchasing_customers,

    SUM(CASE WHEN total_orders = 1 THEN 1 ELSE 0 END) AS one_time_buyers,
    SUM(CASE WHEN total_orders > 1 THEN 1 ELSE 0 END) AS repeat_buyers,

    ROUND(
        100.0 * SUM(CASE WHEN total_orders = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS one_time_buyer_pct,

    ROUND(
        100.0 * SUM(CASE WHEN total_orders > 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS repeat_purchase_rate_pct,

    ROUND(AVG(total_orders), 2) AS avg_orders_per_customer,
    ROUND(AVG(total_revenue), 2) AS avg_revenue_per_purchasing_customer

FROM customer_orders;
"""

con.execute(query).df()

,purchasing_customers,one_time_buyers,repeat_buyers,one_time_buyer_pct,repeat_purchase_rate_pct,avg_orders_per_customer,avg_revenue_per_purchasing_customer
0,193854,10145.0,183709.0,5.23,94.77,9.68,980.86


# **Repeat Purchase by Purchase Days**

In [23]:
query = """
WITH customer_purchase_days AS (
    SELECT
        customer_id,
        COUNT(DISTINCT CAST(transaction_date AS DATE)) AS purchase_days,
        COUNT(DISTINCT DATE_TRUNC('month', transaction_date)) AS purchase_months,
        COUNT(DISTINCT transaction_id) AS transaction_rows,
        SUM(total_amount) AS total_revenue
    FROM transactions_clean
    WHERE status = 'completed'
    GROUP BY customer_id
)

SELECT
    COUNT(*) AS purchasing_customers,

    SUM(CASE WHEN purchase_days = 1 THEN 1 ELSE 0 END) AS one_day_buyers,
    SUM(CASE WHEN purchase_days > 1 THEN 1 ELSE 0 END) AS repeat_day_buyers,

    ROUND(
        100.0 * SUM(CASE WHEN purchase_days = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS one_day_buyer_pct,

    ROUND(
        100.0 * SUM(CASE WHEN purchase_days > 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS repeat_purchase_day_rate_pct,

    SUM(CASE WHEN purchase_months = 1 THEN 1 ELSE 0 END) AS one_month_buyers,
    SUM(CASE WHEN purchase_months > 1 THEN 1 ELSE 0 END) AS repeat_month_buyers,

    ROUND(
        100.0 * SUM(CASE WHEN purchase_months > 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS repeat_purchase_month_rate_pct,

    ROUND(AVG(purchase_days), 2) AS avg_purchase_days_per_customer,
    ROUND(AVG(purchase_months), 2) AS avg_purchase_months_per_customer,
    ROUND(AVG(total_revenue), 2) AS avg_revenue_per_purchasing_customer

FROM customer_purchase_days;
"""

con.execute(query).df()

,purchasing_customers,one_day_buyers,repeat_day_buyers,one_day_buyer_pct,repeat_purchase_day_rate_pct,one_month_buyers,repeat_month_buyers,repeat_purchase_month_rate_pct,avg_purchase_days_per_customer,avg_purchase_months_per_customer,avg_revenue_per_purchasing_customer
0,193854,10149.0,183705.0,5.24,94.76,10356.0,183498.0,94.66,9.64,8.73,980.86


# **Completed Revenue Overview**

In [24]:
query = """
SELECT
    COUNT(*) AS total_transaction_rows,

    SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,
    SUM(CASE WHEN status = 'returned' THEN 1 ELSE 0 END) AS returned_transactions,
    SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_transactions,
    SUM(CASE WHEN status = 'pending' THEN 1 ELSE 0 END) AS pending_transactions,

    ROUND(
        100.0 * SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS completed_transaction_pct,

    ROUND(SUM(CASE WHEN status = 'completed' THEN total_amount ELSE 0 END), 2) AS completed_revenue,

    ROUND(AVG(CASE WHEN status = 'completed' THEN total_amount END), 2) AS avg_completed_transaction_value,

    SUM(CASE WHEN status = 'completed' THEN quantity ELSE 0 END) AS completed_units_sold,

    ROUND(SUM(CASE WHEN status = 'returned' THEN total_amount ELSE 0 END), 2) AS returned_revenue_value,

    ROUND(SUM(CASE WHEN status = 'cancelled' THEN total_amount ELSE 0 END), 2) AS cancelled_revenue_value

FROM transactions_clean;
"""

con.execute(query).df()

,total_transaction_rows,completed_transactions,returned_transactions,cancelled_transactions,pending_transactions,completed_transaction_pct,completed_revenue,avg_completed_transaction_value,completed_units_sold,returned_revenue_value,cancelled_revenue_value
0,3000000,1875600.0,375186.0,374518.0,374696.0,62.52,1.901442e+08,101.38,2907305.0,38037906.57,37978299.95


# **Revenue and Order Status by Segment**

In [25]:
query = """
SELECT
    c.segment,

    COUNT(*) AS total_transaction_rows,

    SUM(CASE WHEN t.status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,
    SUM(CASE WHEN t.status = 'returned' THEN 1 ELSE 0 END) AS returned_transactions,
    SUM(CASE WHEN t.status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_transactions,

    ROUND(
        100.0 * SUM(CASE WHEN t.status = 'completed' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS completion_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN t.status = 'returned' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS return_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN t.status = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS cancellation_rate_pct,

    ROUND(SUM(CASE WHEN t.status = 'completed' THEN t.total_amount ELSE 0 END), 2) AS completed_revenue,

    ROUND(AVG(CASE WHEN t.status = 'completed' THEN t.total_amount END), 2) AS avg_completed_transaction_value,

    COUNT(DISTINCT t.customer_id) AS purchasing_customers,

    ROUND(
        SUM(CASE WHEN t.status = 'completed' THEN t.total_amount ELSE 0 END)
        / COUNT(DISTINCT t.customer_id),
        2
    ) AS revenue_per_purchasing_customer

FROM transactions_clean t
JOIN customers_clean c
    ON t.customer_id = c.customer_id
GROUP BY c.segment
ORDER BY completed_revenue DESC;
"""

con.execute(query).df()

,segment,total_transaction_rows,completed_transactions,returned_transactions,cancelled_transactions,completion_rate_pct,return_rate_pct,cancellation_rate_pct,completed_revenue,avg_completed_transaction_value,purchasing_customers,revenue_per_purchasing_customer
0,UA Rewards Member,1005145,628640.0,125712.0,125131.0,62.54,12.51,12.45,63466877.26,100.96,44240,1434.60
1,Performance Athlete,815162,509986.0,102076.0,101462.0,62.56,12.52,12.45,51976491.23,101.92,39854,1304.17
2,UA Rewards Elite,540148,337170.0,67500.0,68077.0,62.42,12.50,12.60,33965177.57,100.74,15878,2139.13
3,Casual Buyer,445513,278612.0,55610.0,55643.0,62.54,12.48,12.49,28370596.76,101.83,56037,506.28
4,Discount Hunter,170002,106237.0,21243.0,21196.0,62.49,12.50,12.47,10811274.90,101.77,29758,363.31
5,One-Time Buyer,24030,14955.0,3045.0,3009.0,62.23,12.67,12.52,1553827.33,103.90,11541,134.64


# **Monthly Revenue Trend**

In [26]:
query = """
SELECT
    DATE_TRUNC('month', transaction_date) AS month,

    COUNT(*) AS total_transaction_rows,

    SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,

    COUNT(DISTINCT CASE WHEN status = 'completed' THEN customer_id END) AS purchasing_customers,

    ROUND(SUM(CASE WHEN status = 'completed' THEN total_amount ELSE 0 END), 2) AS completed_revenue,

    ROUND(AVG(CASE WHEN status = 'completed' THEN total_amount END), 2) AS avg_completed_transaction_value,

    ROUND(
        100.0 * SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS completion_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN status = 'returned' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS return_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS cancellation_rate_pct

FROM transactions_clean
GROUP BY DATE_TRUNC('month', transaction_date)
ORDER BY month;
"""

con.execute(query).df()

,month,total_transaction_rows,completed_transactions,purchasing_customers,completed_revenue,avg_completed_transaction_value,completion_rate_pct,return_rate_pct,cancellation_rate_pct
0,2021-01-01,49414,30977.0,27860,3495545.23,112.84,62.69,12.76,12.11
1,2021-02-01,44384,27712.0,25221,2248340.33,81.13,62.44,12.69,12.68
2,2021-03-01,49230,30663.0,27628,2681659.40,87.46,62.29,12.60,12.50
3,2021-04-01,47708,29821.0,26920,2688869.06,90.17,62.51,12.69,12.25
4,2021-05-01,49244,30888.0,27886,2873836.83,93.04,62.72,12.76,12.05
...,...,...,...,...,...,...,...,...,...
57,2025-10-01,49518,30862.0,27796,3083712.74,99.92,62.32,12.49,12.66
58,2025-11-01,47655,29770.0,26976,3830019.94,128.65,62.47,12.47,12.46
59,2025-12-01,49128,30709.0,27743,4175859.89,135.98,62.51,12.29,12.54
60,2026-01-01,49363,30929.0,27828,3496911.82,113.06,62.66,12.56,12.37


# **Create Executive Monthly KPI View**

In [27]:
query = """
CREATE OR REPLACE VIEW bi_monthly_revenue_kpis AS
SELECT
    DATE_TRUNC('month', transaction_date) AS month,

    COUNT(*) AS total_transaction_rows,

    SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,
    SUM(CASE WHEN status = 'returned' THEN 1 ELSE 0 END) AS returned_transactions,
    SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_transactions,
    SUM(CASE WHEN status = 'pending' THEN 1 ELSE 0 END) AS pending_transactions,

    COUNT(DISTINCT CASE WHEN status = 'completed' THEN customer_id END) AS purchasing_customers,

    ROUND(SUM(CASE WHEN status = 'completed' THEN total_amount ELSE 0 END), 2) AS completed_revenue,

    ROUND(AVG(CASE WHEN status = 'completed' THEN total_amount END), 2) AS avg_completed_transaction_value,

    SUM(CASE WHEN status = 'completed' THEN quantity ELSE 0 END) AS completed_units_sold,

    ROUND(
        100.0 * SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS completion_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN status = 'returned' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS return_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS cancellation_rate_pct

FROM transactions_clean
GROUP BY DATE_TRUNC('month', transaction_date);
"""

con.execute(query)

print("bi_monthly_revenue_kpis view created")

bi_monthly_revenue_kpis view created


In [28]:
con.execute("SELECT * FROM bi_monthly_revenue_kpis ORDER BY month LIMIT 5").df()

,month,total_transaction_rows,completed_transactions,returned_transactions,cancelled_transactions,pending_transactions,purchasing_customers,completed_revenue,avg_completed_transaction_value,completed_units_sold,completion_rate_pct,return_rate_pct,cancellation_rate_pct
0,2021-01-01,49414,30977.0,6303.0,5984.0,6150.0,27860,3495545.23,112.84,53269.0,62.69,12.76,12.11
1,2021-02-01,44384,27712.0,5634.0,5630.0,5408.0,25221,2248340.33,81.13,34617.0,62.44,12.69,12.68
2,2021-03-01,49230,30663.0,6204.0,6153.0,6210.0,27628,2681659.40,87.46,40650.0,62.29,12.60,12.50
3,2021-04-01,47708,29821.0,6055.0,5845.0,5987.0,26920,2688869.06,90.17,41014.0,62.51,12.69,12.25
4,2021-05-01,49244,30888.0,6282.0,5933.0,6141.0,27886,2873836.83,93.04,43952.0,62.72,12.76,12.05


# **Create Segment Churn + Revenue View**

In [29]:
query = """
CREATE OR REPLACE VIEW bi_segment_churn_revenue AS
WITH segment_customer_base AS (
    SELECT
        segment,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
        SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
        SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) AS loyalty_members,
        ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value
    FROM customers_clean
    GROUP BY segment
),

segment_revenue AS (
    SELECT
        c.segment,
        COUNT(DISTINCT t.customer_id) AS purchasing_customers,
        SUM(CASE WHEN t.status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,
        ROUND(SUM(CASE WHEN t.status = 'completed' THEN t.total_amount ELSE 0 END), 2) AS completed_revenue,
        ROUND(AVG(CASE WHEN t.status = 'completed' THEN t.total_amount END), 2) AS avg_completed_transaction_value
    FROM transactions_clean t
    JOIN customers_clean c
        ON t.customer_id = c.customer_id
    GROUP BY c.segment
)

SELECT
    cb.segment,
    cb.total_customers,
    cb.active_customers,
    cb.churned_customers,

    ROUND(100.0 * cb.churned_customers / cb.total_customers, 2) AS churn_rate_pct,
    ROUND(100.0 * cb.active_customers / cb.total_customers, 2) AS retention_rate_pct,

    cb.loyalty_members,
    ROUND(100.0 * cb.loyalty_members / cb.total_customers, 2) AS loyalty_member_pct,

    cb.avg_lifetime_value,

    COALESCE(sr.purchasing_customers, 0) AS purchasing_customers,
    COALESCE(sr.completed_transactions, 0) AS completed_transactions,
    COALESCE(sr.completed_revenue, 0) AS completed_revenue,
    COALESCE(sr.avg_completed_transaction_value, 0) AS avg_completed_transaction_value,

    ROUND(
        COALESCE(sr.completed_revenue, 0) / cb.total_customers,
        2
    ) AS revenue_per_customer

FROM segment_customer_base cb
LEFT JOIN segment_revenue sr
    ON cb.segment = sr.segment
ORDER BY churn_rate_pct DESC;
"""

con.execute(query)

print("bi_segment_churn_revenue view created")

bi_segment_churn_revenue view created


In [30]:
con.execute("SELECT * FROM bi_segment_churn_revenue").df()

,segment,total_customers,active_customers,churned_customers,churn_rate_pct,retention_rate_pct,loyalty_members,loyalty_member_pct,avg_lifetime_value,purchasing_customers,completed_transactions,completed_revenue,avg_completed_transaction_value,revenue_per_customer
0,One-Time Buyer,14099,6330.0,7769.0,55.10,44.90,4983.0,35.34,656.17,11541,14955.0,1553827.33,103.90,110.21
1,Discount Hunter,29872,20181.0,9691.0,32.44,67.56,10333.0,34.59,781.11,29758,106237.0,10811274.90,101.77,361.92
2,Casual Buyer,56057,43746.0,12311.0,21.96,78.04,19608.0,34.98,1123.41,56037,278612.0,28370596.76,101.83,506.10
3,UA Rewards Member,44240,40289.0,3951.0,8.93,91.07,44240.0,100.00,2764.40,44240,628640.0,63466877.26,100.96,1434.60
4,Performance Athlete,39854,37081.0,2773.0,6.96,93.04,13970.0,35.05,3460.63,39854,509986.0,51976491.23,101.92,1304.17
5,UA Rewards Elite,15878,15242.0,636.0,4.01,95.99,15878.0,100.00,5555.71,15878,337170.0,33965177.57,100.74,2139.13


# **Create Customer RFM Base**

In [31]:
query = """
CREATE OR REPLACE VIEW rfm_base AS
WITH max_date AS (
    SELECT MAX(CAST(transaction_date AS DATE)) AS analysis_date
    FROM transactions_clean
    WHERE status = 'completed'
)

SELECT
    t.customer_id,

    MIN(CAST(t.transaction_date AS DATE)) AS first_purchase_date,
    MAX(CAST(t.transaction_date AS DATE)) AS last_purchase_date,

    DATE_DIFF(
        'day',
        MAX(CAST(t.transaction_date AS DATE)),
        (SELECT analysis_date FROM max_date)
    ) AS recency_days,

    COUNT(DISTINCT t.transaction_id) AS frequency,

    ROUND(SUM(t.total_amount), 2) AS monetary_value,

    ROUND(AVG(t.total_amount), 2) AS avg_order_value

FROM transactions_clean t
WHERE t.status = 'completed'
GROUP BY t.customer_id;
"""

con.execute(query)

print("rfm_base view created")

rfm_base view created


In [32]:
con.execute("SELECT * FROM rfm_base ORDER BY monetary_value DESC LIMIT 10").df()

,customer_id,first_purchase_date,last_purchase_date,recency_days,frequency,monetary_value,avg_order_value
0,UA-C156268,2021-01-01,2025-11-24,95,29,6750.00,232.76
1,UA-C078103,2021-03-25,2026-01-31,27,30,6713.87,223.80
2,UA-C186083,2021-02-07,2025-11-08,111,33,6543.22,198.28
3,UA-C024381,2021-05-05,2025-12-02,87,12,6285.90,523.82
4,UA-C146703,2021-01-31,2026-01-03,55,26,6191.57,238.14
5,UA-C029410,2021-01-11,2026-02-27,0,23,6065.86,263.73
6,UA-C157446,2021-07-03,2025-12-15,74,15,5965.07,397.67
7,UA-C164518,2022-01-29,2026-02-02,25,21,5853.72,278.75
8,UA-C029514,2021-04-10,2025-12-25,64,24,5720.83,238.37
9,UA-C123613,2021-01-22,2026-01-05,53,33,5645.37,171.07


# **Create RFM Scores**

In [33]:
query = """
CREATE OR REPLACE VIEW rfm_scored AS
SELECT
    customer_id,
    first_purchase_date,
    last_purchase_date,
    recency_days,
    frequency,
    monetary_value,
    avg_order_value,

    NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
    NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
    NTILE(5) OVER (ORDER BY monetary_value ASC) AS m_score

FROM rfm_base;
"""

con.execute(query)

print("rfm_scored view created")

rfm_scored view created


In [34]:
query = """
SELECT
    customer_id,
    recency_days,
    frequency,
    monetary_value,
    r_score,
    f_score,
    m_score,
    CAST(r_score AS VARCHAR) || CAST(f_score AS VARCHAR) || CAST(m_score AS VARCHAR) AS rfm_score
FROM rfm_scored
ORDER BY r_score DESC, f_score DESC, m_score DESC
LIMIT 10;
"""

con.execute(query).df()

,customer_id,recency_days,frequency,monetary_value,r_score,f_score,m_score,rfm_score
0,UA-C140822,27,18,1589.48,5,5,5,555
1,UA-C178217,14,20,1589.59,5,5,5,555
2,UA-C191203,29,20,1589.59,5,5,5,555
3,UA-C189463,23,16,1589.64,5,5,5,555
4,UA-C043293,4,16,1589.96,5,5,5,555
5,UA-C061579,7,18,1589.59,5,5,5,555
6,UA-C189797,13,21,1590.02,5,5,5,555
7,UA-C012975,29,18,1589.64,5,5,5,555
8,UA-C102397,3,20,1589.84,5,5,5,555
9,UA-C010351,4,19,1590.14,5,5,5,555


# **Create RFM Customer Segments**

In [35]:
query = """
CREATE OR REPLACE VIEW rfm_segments AS
SELECT
    customer_id,
    first_purchase_date,
    last_purchase_date,
    recency_days,
    frequency,
    monetary_value,
    avg_order_value,
    r_score,
    f_score,
    m_score,
    CAST(r_score AS VARCHAR) || CAST(f_score AS VARCHAR) || CAST(m_score AS VARCHAR) AS rfm_score,

    CASE
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4
            THEN 'VIP Customers'

        WHEN r_score >= 4 AND f_score >= 3
            THEN 'Loyal Customers'

        WHEN r_score >= 4 AND f_score <= 2
            THEN 'New / Promising Customers'

        WHEN r_score <= 2 AND f_score >= 4 AND m_score >= 4
            THEN 'At-Risk High-Value Customers'

        WHEN r_score <= 2 AND f_score >= 3
            THEN 'At-Risk Loyal Customers'

        WHEN r_score <= 2 AND f_score <= 2
            THEN 'Lost / Dormant Customers'

        WHEN m_score >= 4 AND f_score <= 2
            THEN 'Big Spenders - Low Frequency'

        ELSE 'Regular Customers'
    END AS rfm_segment

FROM rfm_scored;
"""

con.execute(query)

print("rfm_segments view created")

rfm_segments view created


In [36]:
query = """
SELECT
    rfm_segment,
    COUNT(*) AS customers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS customer_pct,
    ROUND(AVG(recency_days), 2) AS avg_recency_days,
    ROUND(AVG(frequency), 2) AS avg_frequency,
    ROUND(AVG(monetary_value), 2) AS avg_monetary_value,
    ROUND(SUM(monetary_value), 2) AS total_monetary_value
FROM rfm_segments
GROUP BY rfm_segment
ORDER BY total_monetary_value DESC;
"""

con.execute(query).df()

,rfm_segment,customers,customer_pct,avg_recency_days,avg_frequency,avg_monetary_value,total_monetary_value
0,VIP Customers,41799,21.56,44.69,17.44,1843.96,77075594.40
1,Regular Customers,38403,19.81,156.88,10.76,1074.97,41282249.51
2,Loyal Customers,22581,11.65,48.26,9.83,878.90,19846520.08
3,Lost / Dormant Customers,52378,27.02,677.37,3.22,324.73,17008488.72
4,At-Risk High-Value Customers,9313,4.80,323.88,15.49,1656.08,15423030.06
5,At-Risk Loyal Customers,15851,8.18,380.32,9.10,852.28,13509426.38
6,New / Promising Customers,13161,6.79,56.19,3.96,417.97,5500874.22
7,Big Spenders - Low Frequency,368,0.19,156.96,5.16,1353.43,498061.68


# **Create BI RFM Customer Segments View**

In [37]:
query = """
CREATE OR REPLACE VIEW bi_rfm_customer_segments AS
SELECT
    r.customer_id,
    c.segment AS original_customer_segment,
    r.rfm_segment,
    r.rfm_score,

    r.first_purchase_date,
    r.last_purchase_date,
    r.recency_days,
    r.frequency,
    r.monetary_value,
    r.avg_order_value,

    r.r_score,
    r.f_score,
    r.m_score,

    c.country,
    c.gender,
    c.age,
    c.is_churned,
    c.is_loyalty_member,
    c.ua_rewards_points,
    c.email_opt_in,
    c.has_app,
    c.preferred_sport,
    c.lifetime_value,

    CASE
        WHEN r.rfm_segment IN ('At-Risk High-Value Customers', 'At-Risk Loyal Customers')
            THEN 'Win Back'

        WHEN r.rfm_segment = 'VIP Customers'
            THEN 'Protect and Reward'

        WHEN r.rfm_segment = 'Lost / Dormant Customers'
            THEN 'Reactivation'

        WHEN r.rfm_segment = 'New / Promising Customers'
            THEN 'Nurture'

        WHEN r.rfm_segment = 'Big Spenders - Low Frequency'
            THEN 'Increase Frequency'

        ELSE 'Maintain Engagement'
    END AS recommended_action

FROM rfm_segments r
JOIN customers_clean c
    ON r.customer_id = c.customer_id;
"""

con.execute(query)

print("bi_rfm_customer_segments view created")

bi_rfm_customer_segments view created


In [38]:
query = """
SELECT
    rfm_segment,
    recommended_action,
    COUNT(*) AS customers,
    ROUND(AVG(recency_days), 2) AS avg_recency_days,
    ROUND(AVG(frequency), 2) AS avg_frequency,
    ROUND(AVG(monetary_value), 2) AS avg_monetary_value,
    ROUND(SUM(monetary_value), 2) AS total_monetary_value,
    ROUND(100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loyalty_member_pct
FROM bi_rfm_customer_segments
GROUP BY rfm_segment, recommended_action
ORDER BY total_monetary_value DESC;
"""

con.execute(query).df()

,rfm_segment,recommended_action,customers,avg_recency_days,avg_frequency,avg_monetary_value,total_monetary_value,churn_rate_pct,loyalty_member_pct
0,VIP Customers,Protect and Reward,42201,44.93,17.38,1840.05,77651816.75,7.37,79.69
1,Regular Customers,Maintain Engagement,38401,156.88,10.76,1074.79,41272837.09,15.00,58.03
2,Loyal Customers,Maintain Engagement,23055,48.82,9.67,860.61,19841343.89,13.32,54.89
3,Lost / Dormant Customers,Reactivation,53248,672.82,3.27,329.43,17541698.70,30.19,35.33
4,At-Risk High-Value Customers,Win Back,8965,318.20,15.66,1665.35,14929865.46,7.16,75.38
5,At-Risk Loyal Customers,Win Back,15329,381.30,9.32,879.07,13475215.44,13.69,54.03
6,New / Promising Customers,Nurture,12285,55.02,3.81,401.39,4931072.26,26.76,34.91
7,Big Spenders - Low Frequency,Increase Frequency,370,156.99,5.16,1352.42,500395.46,21.08,32.70


# **Recreate Stable RFM Scoring View**

In [39]:
query = """
CREATE OR REPLACE VIEW rfm_scored AS
SELECT
    customer_id,
    first_purchase_date,
    last_purchase_date,
    recency_days,
    frequency,
    monetary_value,
    avg_order_value,

    NTILE(5) OVER (
        ORDER BY recency_days DESC, customer_id
    ) AS r_score,

    NTILE(5) OVER (
        ORDER BY frequency ASC, customer_id
    ) AS f_score,

    NTILE(5) OVER (
        ORDER BY monetary_value ASC, customer_id
    ) AS m_score

FROM rfm_base;
"""

con.execute(query)

print("stable rfm_scored view created")

stable rfm_scored view created


# **Recreate Stable RFM Segments**

In [40]:
query = """
CREATE OR REPLACE VIEW rfm_segments AS
SELECT
    customer_id,
    first_purchase_date,
    last_purchase_date,
    recency_days,
    frequency,
    monetary_value,
    avg_order_value,
    r_score,
    f_score,
    m_score,
    CAST(r_score AS VARCHAR) || CAST(f_score AS VARCHAR) || CAST(m_score AS VARCHAR) AS rfm_score,

    CASE
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4
            THEN 'VIP Customers'

        WHEN r_score >= 4 AND f_score >= 3
            THEN 'Loyal Customers'

        WHEN r_score >= 4 AND f_score <= 2
            THEN 'New / Promising Customers'

        WHEN r_score <= 2 AND f_score >= 4 AND m_score >= 4
            THEN 'At-Risk High-Value Customers'

        WHEN r_score <= 2 AND f_score >= 3
            THEN 'At-Risk Loyal Customers'

        WHEN r_score <= 2 AND f_score <= 2
            THEN 'Lost / Dormant Customers'

        WHEN m_score >= 4 AND f_score <= 2
            THEN 'Big Spenders - Low Frequency'

        ELSE 'Regular Customers'
    END AS rfm_segment

FROM rfm_scored;
"""

con.execute(query)

print("stable rfm_segments view created")

stable rfm_segments view created


# **recreate the BI-ready view:**

In [41]:
query = """
CREATE OR REPLACE VIEW bi_rfm_customer_segments AS
SELECT
    r.customer_id,
    c.segment AS original_customer_segment,
    r.rfm_segment,
    r.rfm_score,

    r.first_purchase_date,
    r.last_purchase_date,
    r.recency_days,
    r.frequency,
    r.monetary_value,
    r.avg_order_value,

    r.r_score,
    r.f_score,
    r.m_score,

    c.country,
    c.gender,
    c.age,
    c.is_churned,
    c.is_loyalty_member,
    c.ua_rewards_points,
    c.email_opt_in,
    c.has_app,
    c.preferred_sport,
    c.lifetime_value,

    CASE
        WHEN r.rfm_segment IN ('At-Risk High-Value Customers', 'At-Risk Loyal Customers')
            THEN 'Win Back'

        WHEN r.rfm_segment = 'VIP Customers'
            THEN 'Protect and Reward'

        WHEN r.rfm_segment = 'Lost / Dormant Customers'
            THEN 'Reactivation'

        WHEN r.rfm_segment = 'New / Promising Customers'
            THEN 'Nurture'

        WHEN r.rfm_segment = 'Big Spenders - Low Frequency'
            THEN 'Increase Frequency'

        ELSE 'Maintain Engagement'
    END AS recommended_action

FROM rfm_segments r
JOIN customers_clean c
    ON r.customer_id = c.customer_id;
"""

con.execute(query)

print("stable bi_rfm_customer_segments view created")

stable bi_rfm_customer_segments view created


In [42]:
query = """
SELECT
    rfm_segment,
    recommended_action,
    COUNT(*) AS customers,
    ROUND(AVG(recency_days), 2) AS avg_recency_days,
    ROUND(AVG(frequency), 2) AS avg_frequency,
    ROUND(AVG(monetary_value), 2) AS avg_monetary_value,
    ROUND(SUM(monetary_value), 2) AS total_monetary_value,
    ROUND(100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loyalty_member_pct
FROM bi_rfm_customer_segments
GROUP BY rfm_segment, recommended_action
ORDER BY total_monetary_value DESC;
"""

con.execute(query).df()

,rfm_segment,recommended_action,customers,avg_recency_days,avg_frequency,avg_monetary_value,total_monetary_value,churn_rate_pct,loyalty_member_pct
0,VIP Customers,Protect and Reward,41345,45.30,17.52,1848.66,76432800.56,7.37,80.04
1,Regular Customers,Maintain Engagement,38439,156.88,10.75,1075.32,41334068.15,14.99,58.01
2,Loyal Customers,Maintain Engagement,21900,49.09,10.05,905.14,19822627.54,12.23,56.75
3,Lost / Dormant Customers,Reactivation,51862,678.70,3.20,322.24,16712201.83,30.42,35.28
4,At-Risk High-Value Customers,Win Back,9345,320.05,15.47,1655.03,15466285.00,7.17,74.87
5,At-Risk Loyal Customers,Win Back,16335,387.78,9.00,842.61,13764039.04,14.48,52.37
6,New / Promising Customers,Nurture,14296,52.44,4.12,431.06,6162419.83,26.25,35.32
7,Big Spenders - Low Frequency,Increase Frequency,332,157.23,5.08,1354.83,449803.10,20.78,33.13


# **Create BI RFM Segment Summary View**

In [43]:
query = """
CREATE OR REPLACE VIEW bi_rfm_segment_summary AS
SELECT
    rfm_segment,
    recommended_action,

    COUNT(*) AS customers,

    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS customer_pct,

    ROUND(AVG(recency_days), 2) AS avg_recency_days,
    ROUND(AVG(frequency), 2) AS avg_frequency,
    ROUND(AVG(monetary_value), 2) AS avg_monetary_value,
    ROUND(AVG(avg_order_value), 2) AS avg_order_value,

    ROUND(SUM(monetary_value), 2) AS total_monetary_value,

    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,

    ROUND(
        100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS retention_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS loyalty_member_pct,

    ROUND(
        100.0 * SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS app_user_pct,

    ROUND(
        100.0 * SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS email_opt_in_pct

FROM bi_rfm_customer_segments
GROUP BY rfm_segment, recommended_action;
"""

con.execute(query)

print("bi_rfm_segment_summary view created")

bi_rfm_segment_summary view created


In [44]:
con.execute("""
SELECT *
FROM bi_rfm_segment_summary
ORDER BY total_monetary_value DESC
""").df()

,rfm_segment,recommended_action,customers,customer_pct,avg_recency_days,avg_frequency,avg_monetary_value,avg_order_value,total_monetary_value,churned_customers,active_customers,churn_rate_pct,retention_rate_pct,loyalty_member_pct,app_user_pct,email_opt_in_pct
0,VIP Customers,Protect and Reward,41345,21.33,45.30,17.52,1848.66,106.85,76432800.56,3048.0,38297.0,7.37,92.63,80.04,61.53,71.87
1,Regular Customers,Maintain Engagement,38439,19.83,156.88,10.75,1075.32,99.40,41334068.15,5763.0,32676.0,14.99,85.01,58.01,61.91,71.99
2,Loyal Customers,Maintain Engagement,21900,11.30,49.09,10.05,905.14,93.47,19822627.54,2678.0,19222.0,12.23,87.77,56.75,62.13,72.10
3,Lost / Dormant Customers,Reactivation,51862,26.75,678.70,3.20,322.24,100.82,16712201.83,15776.0,36086.0,30.42,69.58,35.28,62.09,71.84
4,At-Risk High-Value Customers,Win Back,9345,4.82,320.05,15.47,1655.03,108.39,15466285.00,670.0,8675.0,7.17,92.83,74.87,61.70,72.13
5,At-Risk Loyal Customers,Win Back,16335,8.43,387.78,9.00,842.61,95.76,13764039.04,2365.0,13970.0,14.48,85.52,52.37,61.51,72.12
6,New / Promising Customers,Nurture,14296,7.37,52.44,4.12,431.06,105.53,6162419.83,3753.0,10543.0,26.25,73.75,35.32,62.49,72.53
7,Big Spenders - Low Frequency,Increase Frequency,332,0.17,157.23,5.08,1354.83,283.16,449803.10,69.0,263.0,20.78,79.22,33.13,59.34,71.39


# **Create First Purchase Cohort Base**

In [45]:
query = """
CREATE OR REPLACE VIEW cohort_base AS
WITH first_purchase AS (
    SELECT
        customer_id,
        DATE_TRUNC('month', MIN(transaction_date)) AS cohort_month
    FROM transactions_clean
    WHERE status = 'completed'
    GROUP BY customer_id
),

customer_purchase_months AS (
    SELECT DISTINCT
        customer_id,
        DATE_TRUNC('month', transaction_date) AS purchase_month
    FROM transactions_clean
    WHERE status = 'completed'
)

SELECT
    fp.customer_id,
    fp.cohort_month,
    cpm.purchase_month,

    DATE_DIFF(
        'month',
        fp.cohort_month,
        cpm.purchase_month
    ) AS cohort_age_month

FROM first_purchase fp
JOIN customer_purchase_months cpm
    ON fp.customer_id = cpm.customer_id
WHERE cpm.purchase_month >= fp.cohort_month;
"""

con.execute(query)

print("cohort_base view created")

cohort_base view created


In [46]:
con.execute("""
SELECT *
FROM cohort_base
ORDER BY customer_id, cohort_age_month
LIMIT 20
""").df()

,customer_id,cohort_month,purchase_month,cohort_age_month
0,UA-C000000,2022-07-01,2022-07-01,0
1,UA-C000000,2022-07-01,2024-02-01,19
2,UA-C000000,2022-07-01,2024-11-01,28
3,UA-C000000,2022-07-01,2025-06-01,35
4,UA-C000001,2021-02-01,2021-02-01,0
5,UA-C000001,2021-02-01,2021-03-01,1
6,UA-C000001,2021-02-01,2021-04-01,2
7,UA-C000001,2021-02-01,2021-06-01,4
8,UA-C000001,2021-02-01,2021-10-01,8
9,UA-C000001,2021-02-01,2021-12-01,10


# **Create Cohort Retention Summary**

In [47]:
query = """
CREATE OR REPLACE VIEW bi_cohort_retention_summary AS
WITH cohort_counts AS (
    SELECT
        cohort_month,
        COUNT(DISTINCT customer_id) AS cohort_size
    FROM cohort_base
    WHERE cohort_age_month = 0
    GROUP BY cohort_month
),

retention_counts AS (
    SELECT
        cohort_month,
        cohort_age_month,
        COUNT(DISTINCT customer_id) AS retained_customers
    FROM cohort_base
    GROUP BY cohort_month, cohort_age_month
)

SELECT
    rc.cohort_month,
    rc.cohort_age_month,
    cc.cohort_size,
    rc.retained_customers,

    ROUND(
        100.0 * rc.retained_customers / cc.cohort_size,
        2
    ) AS retention_rate_pct

FROM retention_counts rc
JOIN cohort_counts cc
    ON rc.cohort_month = cc.cohort_month
ORDER BY rc.cohort_month, rc.cohort_age_month;
"""

con.execute(query)

print("bi_cohort_retention_summary view created")

bi_cohort_retention_summary view created


In [48]:
con.execute("""
SELECT *
FROM bi_cohort_retention_summary
WHERE cohort_month <= DATE '2021-06-01'
ORDER BY cohort_month, cohort_age_month
LIMIT 40
""").df()

,cohort_month,cohort_age_month,cohort_size,retained_customers,retention_rate_pct
0,2021-01-01,0,27860,27860,100.00
1,2021-01-01,1,27860,4738,17.01
2,2021-01-01,2,27860,5065,18.18
3,2021-01-01,3,27860,4967,17.83
4,2021-01-01,4,27860,5123,18.39
5,2021-01-01,5,27860,5069,18.19
6,2021-01-01,6,27860,5246,18.83
7,2021-01-01,7,27860,5101,18.31
8,2021-01-01,8,27860,5136,18.44
9,2021-01-01,9,27860,5226,18.76


# **Create Cohort Milestone Summary**

In [49]:
query = """
CREATE OR REPLACE VIEW bi_cohort_milestone_summary AS
SELECT
    cohort_month,

    MAX(CASE WHEN cohort_age_month = 0 THEN cohort_size END) AS cohort_size,

    MAX(CASE WHEN cohort_age_month = 1 THEN retention_rate_pct END) AS month_1_retention_pct,
    MAX(CASE WHEN cohort_age_month = 3 THEN retention_rate_pct END) AS month_3_retention_pct,
    MAX(CASE WHEN cohort_age_month = 6 THEN retention_rate_pct END) AS month_6_retention_pct,
    MAX(CASE WHEN cohort_age_month = 12 THEN retention_rate_pct END) AS month_12_retention_pct,

    MAX(CASE WHEN cohort_age_month = 1 THEN retained_customers END) AS month_1_retained_customers,
    MAX(CASE WHEN cohort_age_month = 3 THEN retained_customers END) AS month_3_retained_customers,
    MAX(CASE WHEN cohort_age_month = 6 THEN retained_customers END) AS month_6_retained_customers,
    MAX(CASE WHEN cohort_age_month = 12 THEN retained_customers END) AS month_12_retained_customers

FROM bi_cohort_retention_summary
GROUP BY cohort_month
ORDER BY cohort_month;
"""

con.execute(query)

print("bi_cohort_milestone_summary view created")

bi_cohort_milestone_summary view created


In [50]:
con.execute("""
SELECT *
FROM bi_cohort_milestone_summary
ORDER BY cohort_month
LIMIT 10
""").df()

,cohort_month,cohort_size,month_1_retention_pct,month_3_retention_pct,month_6_retention_pct,month_12_retention_pct,month_1_retained_customers,month_3_retained_customers,month_6_retained_customers,month_12_retained_customers
0,2021-01-01,27860,17.01,17.83,18.83,18.88,4738,4967,5246,5261
1,2021-02-01,20483,17.73,18.41,17.71,16.13,3632,3771,3627,3303
2,2021-03-01,18931,17.19,17.02,17.28,16.88,3254,3222,3271,3195
3,2021-04-01,15114,16.65,16.83,16.76,16.10,2517,2544,2533,2433
4,2021-05-01,13128,15.90,16.25,15.53,16.12,2088,2133,2039,2116
5,2021-06-01,10767,15.30,14.67,15.53,14.91,1647,1580,1672,1605
6,2021-07-01,9263,14.78,14.62,14.80,14.89,1369,1354,1371,1379
7,2021-08-01,7881,14.02,13.42,12.66,14.55,1105,1058,998,1147
8,2021-09-01,6572,13.88,13.88,13.59,12.78,912,912,893,840
9,2021-10-01,6000,11.97,12.58,12.82,12.55,718,755,769,753


# **Recommendation Model Performance**

In [51]:
query = """
CREATE OR REPLACE VIEW bi_recommendation_model_performance AS
SELECT
    recommendation_model,

    COUNT(*) AS impressions,

    SUM(clicked) AS clicks,
    SUM(purchased) AS purchases,

    ROUND(
        100.0 * SUM(clicked) / COUNT(*),
        2
    ) AS ctr_pct,

    ROUND(
        100.0 * SUM(purchased) / NULLIF(SUM(clicked), 0),
        2
    ) AS cvr_after_click_pct,

    ROUND(
        100.0 * SUM(purchased) / COUNT(*),
        2
    ) AS purchase_rate_per_impression_pct,

    ROUND(SUM(revenue), 2) AS recommendation_revenue,

    ROUND(
        SUM(revenue) / COUNT(*),
        4
    ) AS revenue_per_impression,

    ROUND(
        SUM(revenue) / NULLIF(SUM(clicked), 0),
        2
    ) AS revenue_per_click

FROM recommendation_events_clean
GROUP BY recommendation_model
ORDER BY recommendation_revenue DESC;
"""

con.execute(query)

print("bi_recommendation_model_performance view created")

bi_recommendation_model_performance view created


In [52]:
con.execute("""
SELECT *
FROM bi_recommendation_model_performance
ORDER BY recommendation_revenue DESC
""").df()

,recommendation_model,impressions,clicks,purchases,ctr_pct,cvr_after_click_pct,purchase_rate_per_impression_pct,recommendation_revenue,revenue_per_impression,revenue_per_click
0,personalized,209743,40984.0,4061.0,19.54,9.91,1.94,277354.05,1.3224,6.77
1,sport_based,150153,24943.0,2045.0,16.61,8.20,1.36,140246.94,0.9340,5.62
2,loyalty_based,90152,13343.0,921.0,14.80,6.90,1.02,63735.56,0.7070,4.78
3,generic,149952,14646.0,582.0,9.77,3.97,0.39,39059.15,0.2605,2.67


# **Recommendation Performance by Customer Segment**

In [53]:
query = """
CREATE OR REPLACE VIEW bi_recommendation_segment_performance AS
SELECT
    c.segment,
    re.recommendation_model,

    COUNT(*) AS impressions,
    SUM(re.clicked) AS clicks,
    SUM(re.purchased) AS purchases,

    ROUND(
        100.0 * SUM(re.clicked) / COUNT(*),
        2
    ) AS ctr_pct,

    ROUND(
        100.0 * SUM(re.purchased) / NULLIF(SUM(re.clicked), 0),
        2
    ) AS cvr_after_click_pct,

    ROUND(
        100.0 * SUM(re.purchased) / COUNT(*),
        2
    ) AS purchase_rate_per_impression_pct,

    ROUND(SUM(re.revenue), 2) AS recommendation_revenue,

    ROUND(
        SUM(re.revenue) / COUNT(*),
        4
    ) AS revenue_per_impression

FROM recommendation_events_clean re
JOIN customers_clean c
    ON re.customer_id = c.customer_id
GROUP BY
    c.segment,
    re.recommendation_model;
"""

con.execute(query)

print("bi_recommendation_segment_performance view created")

bi_recommendation_segment_performance view created


In [54]:
con.execute("""
SELECT *
FROM bi_recommendation_segment_performance
ORDER BY segment, recommendation_revenue DESC
""").df()

,segment,recommendation_model,impressions,clicks,purchases,ctr_pct,cvr_after_click_pct,purchase_rate_per_impression_pct,recommendation_revenue,revenue_per_impression
0,Casual Buyer,personalized,59145,11199.0,1115.0,18.93,9.96,1.89,77392.23,1.3085
1,Casual Buyer,sport_based,41766,6814.0,590.0,16.31,8.66,1.41,40789.17,0.9766
2,Casual Buyer,loyalty_based,25322,3542.0,234.0,13.99,6.61,0.92,15675.90,0.6191
3,Casual Buyer,generic,41997,3795.0,171.0,9.04,4.51,0.41,11352.15,0.2703
4,Discount Hunter,personalized,31394,5850.0,564.0,18.63,9.64,1.80,38636.29,1.2307
5,Discount Hunter,sport_based,22395,3540.0,283.0,15.81,7.99,1.26,20247.40,0.9041
6,Discount Hunter,loyalty_based,13425,1917.0,147.0,14.28,7.67,1.09,9235.44,0.6879
7,Discount Hunter,generic,22141,1982.0,85.0,8.95,4.29,0.38,5663.89,0.2558
8,One-Time Buyer,personalized,14685,2730.0,264.0,18.59,9.67,1.80,18130.70,1.2346
9,One-Time Buyer,sport_based,10757,1736.0,130.0,16.14,7.49,1.21,9682.67,0.9001


# **Personalized vs Generic Uplift by Segment**

In [55]:
query = """
CREATE OR REPLACE VIEW bi_personalized_vs_generic_uplift AS
WITH model_perf AS (
    SELECT
        c.segment,
        re.recommendation_model,

        COUNT(*) AS impressions,
        SUM(re.clicked) AS clicks,
        SUM(re.purchased) AS purchases,
        SUM(re.revenue) AS revenue,

        1.0 * SUM(re.clicked) / COUNT(*) AS ctr,
        1.0 * SUM(re.purchased) / COUNT(*) AS purchase_rate,
        1.0 * SUM(re.revenue) / COUNT(*) AS revenue_per_impression

    FROM recommendation_events_clean re
    JOIN customers_clean c
        ON re.customer_id = c.customer_id
    WHERE re.recommendation_model IN ('personalized', 'generic')
    GROUP BY c.segment, re.recommendation_model
),

pivoted AS (
    SELECT
        segment,

        MAX(CASE WHEN recommendation_model = 'personalized' THEN impressions END) AS personalized_impressions,
        MAX(CASE WHEN recommendation_model = 'generic' THEN impressions END) AS generic_impressions,

        MAX(CASE WHEN recommendation_model = 'personalized' THEN ctr END) AS personalized_ctr,
        MAX(CASE WHEN recommendation_model = 'generic' THEN ctr END) AS generic_ctr,

        MAX(CASE WHEN recommendation_model = 'personalized' THEN purchase_rate END) AS personalized_purchase_rate,
        MAX(CASE WHEN recommendation_model = 'generic' THEN purchase_rate END) AS generic_purchase_rate,

        MAX(CASE WHEN recommendation_model = 'personalized' THEN revenue_per_impression END) AS personalized_revenue_per_impression,
        MAX(CASE WHEN recommendation_model = 'generic' THEN revenue_per_impression END) AS generic_revenue_per_impression,

        MAX(CASE WHEN recommendation_model = 'personalized' THEN revenue END) AS personalized_revenue,
        MAX(CASE WHEN recommendation_model = 'generic' THEN revenue END) AS generic_revenue

    FROM model_perf
    GROUP BY segment
)

SELECT
    segment,

    personalized_impressions,
    generic_impressions,

    ROUND(100 * personalized_ctr, 2) AS personalized_ctr_pct,
    ROUND(100 * generic_ctr, 2) AS generic_ctr_pct,

    ROUND(
        100.0 * (personalized_ctr - generic_ctr) / NULLIF(generic_ctr, 0),
        2
    ) AS ctr_uplift_pct,

    ROUND(100 * personalized_purchase_rate, 2) AS personalized_purchase_rate_pct,
    ROUND(100 * generic_purchase_rate, 2) AS generic_purchase_rate_pct,

    ROUND(
        100.0 * (personalized_purchase_rate - generic_purchase_rate) / NULLIF(generic_purchase_rate, 0),
        2
    ) AS purchase_rate_uplift_pct,

    ROUND(personalized_revenue_per_impression, 4) AS personalized_revenue_per_impression,
    ROUND(generic_revenue_per_impression, 4) AS generic_revenue_per_impression,

    ROUND(
        100.0 * (personalized_revenue_per_impression - generic_revenue_per_impression)
        / NULLIF(generic_revenue_per_impression, 0),
        2
    ) AS revenue_per_impression_uplift_pct,

    ROUND(personalized_revenue, 2) AS personalized_revenue,
    ROUND(generic_revenue, 2) AS generic_revenue

FROM pivoted
ORDER BY revenue_per_impression_uplift_pct DESC;
"""

con.execute(query)

print("bi_personalized_vs_generic_uplift view created")

bi_personalized_vs_generic_uplift view created


In [56]:
con.execute("""
SELECT *
FROM bi_personalized_vs_generic_uplift
ORDER BY revenue_per_impression_uplift_pct DESC
""").df()

,segment,personalized_impressions,generic_impressions,personalized_ctr_pct,generic_ctr_pct,ctr_uplift_pct,personalized_purchase_rate_pct,generic_purchase_rate_pct,purchase_rate_uplift_pct,personalized_revenue_per_impression,generic_revenue_per_impression,revenue_per_impression_uplift_pct,personalized_revenue,generic_revenue
0,One-Time Buyer,14685,10530,18.59,9.29,100.16,1.80,0.33,440.87,1.2346,0.1868,561.11,18130.70,1966.52
1,Performance Athlete,41354,29955,19.01,9.23,105.84,1.83,0.31,490.39,1.2308,0.2203,458.65,50897.43,6599.48
2,UA Rewards Member,46685,33428,21.20,11.33,86.99,2.16,0.42,407.78,1.4801,0.2909,408.87,69097.79,9722.71
3,Casual Buyer,59145,41997,18.93,9.04,109.54,1.89,0.41,363.00,1.3085,0.2703,384.08,77392.23,11352.15
4,Discount Hunter,31394,22141,18.63,8.95,108.16,1.80,0.38,367.96,1.2307,0.2558,381.10,38636.29,5663.89
5,UA Rewards Elite,16480,11901,20.93,11.23,86.48,2.14,0.47,355.21,1.4077,0.3155,346.24,23199.61,3754.40


# **A/B Test Performance by Experiment**

In [57]:
query = """
CREATE OR REPLACE VIEW bi_experiment_ab_summary AS
SELECT
    experiment_id,
    experiment_name,
    experiment_goal,
    variant,

    COUNT(*) AS exposed_users,

    SUM(clicked_recommendation) AS clicks,
    SUM(converted) AS conversions,

    ROUND(
        100.0 * SUM(clicked_recommendation) / COUNT(*),
        2
    ) AS ctr_pct,

    ROUND(
        100.0 * SUM(converted) / COUNT(*),
        2
    ) AS conversion_rate_pct,

    ROUND(SUM(revenue), 2) AS revenue,

    ROUND(
        SUM(revenue) / COUNT(*),
        2
    ) AS revenue_per_exposed_user

FROM experiments_clean
GROUP BY
    experiment_id,
    experiment_name,
    experiment_goal,
    variant;
"""

con.execute(query)

print("bi_experiment_ab_summary view created")

bi_experiment_ab_summary view created


In [58]:
con.execute("""
SELECT *
FROM bi_experiment_ab_summary
ORDER BY experiment_id, variant
""").df()

,experiment_id,experiment_name,experiment_goal,variant,exposed_users,clicks,conversions,ctr_pct,conversion_rate_pct,revenue,revenue_per_exposed_user
0,EXP001,Personalized vs Generic Homepage Recommendations,increase_recommendation_ctr,control,16577,2061.0,847.0,12.43,5.11,145103.86,8.75
1,EXP001,Personalized vs Generic Homepage Recommendations,increase_recommendation_ctr,treatment,16755,3261.0,1383.0,19.46,8.25,217855.87,13.00
2,EXP002,UA Rewards Loyalty Banner Test,increase_loyalty_member_conversion,control,16614,1981.0,800.0,11.92,4.82,138473.38,8.33
3,EXP002,UA Rewards Loyalty Banner Test,increase_loyalty_member_conversion,treatment,16719,2595.0,1200.0,15.52,7.18,210948.43,12.62
4,EXP003,Sport-Based Recommendation Engine v1,increase_recommendation_ctr,control,16551,2059.0,849.0,12.44,5.13,131506.50,7.95
5,EXP003,Sport-Based Recommendation Engine v1,increase_recommendation_ctr,treatment,16782,2745.0,1638.0,16.36,9.76,258063.00,15.38
6,EXP004,New Homepage Layout vs Old Layout,improve_homepage_conversion,control,16631,1979.0,877.0,11.90,5.27,137176.71,8.25
7,EXP004,New Homepage Layout vs Old Layout,improve_homepage_conversion,treatment,16702,3203.0,1564.0,19.18,9.36,252984.37,15.15
8,EXP005,Search Ranking Algorithm A vs B,improve_search_conversion,control,16465,1969.0,838.0,11.96,5.09,139932.27,8.50
9,EXP005,Search Ranking Algorithm A vs B,improve_search_conversion,treatment,16868,2994.0,1278.0,17.75,7.58,200872.82,11.91


# **Create A/B Test Uplift Summary**

In [59]:
query = """
CREATE OR REPLACE VIEW bi_experiment_uplift_summary AS
WITH pivoted AS (
    SELECT
        experiment_id,
        experiment_name,
        experiment_goal,

        MAX(CASE WHEN variant = 'control' THEN exposed_users END) AS control_users,
        MAX(CASE WHEN variant = 'treatment' THEN exposed_users END) AS treatment_users,

        MAX(CASE WHEN variant = 'control' THEN ctr_pct END) AS control_ctr_pct,
        MAX(CASE WHEN variant = 'treatment' THEN ctr_pct END) AS treatment_ctr_pct,

        MAX(CASE WHEN variant = 'control' THEN conversion_rate_pct END) AS control_conversion_rate_pct,
        MAX(CASE WHEN variant = 'treatment' THEN conversion_rate_pct END) AS treatment_conversion_rate_pct,

        MAX(CASE WHEN variant = 'control' THEN revenue END) AS control_revenue,
        MAX(CASE WHEN variant = 'treatment' THEN revenue END) AS treatment_revenue,

        MAX(CASE WHEN variant = 'control' THEN revenue_per_exposed_user END) AS control_revenue_per_user,
        MAX(CASE WHEN variant = 'treatment' THEN revenue_per_exposed_user END) AS treatment_revenue_per_user

    FROM bi_experiment_ab_summary
    GROUP BY experiment_id, experiment_name, experiment_goal
)

SELECT
    experiment_id,
    experiment_name,
    experiment_goal,

    control_users,
    treatment_users,

    control_ctr_pct,
    treatment_ctr_pct,

    ROUND(
        treatment_ctr_pct - control_ctr_pct,
        2
    ) AS ctr_lift_pp,

    ROUND(
        100.0 * (treatment_ctr_pct - control_ctr_pct) / NULLIF(control_ctr_pct, 0),
        2
    ) AS ctr_uplift_pct,

    control_conversion_rate_pct,
    treatment_conversion_rate_pct,

    ROUND(
        treatment_conversion_rate_pct - control_conversion_rate_pct,
        2
    ) AS conversion_lift_pp,

    ROUND(
        100.0 * (treatment_conversion_rate_pct - control_conversion_rate_pct) / NULLIF(control_conversion_rate_pct, 0),
        2
    ) AS conversion_uplift_pct,

    control_revenue,
    treatment_revenue,

    ROUND(treatment_revenue - control_revenue, 2) AS incremental_revenue,

    control_revenue_per_user,
    treatment_revenue_per_user,

    ROUND(
        treatment_revenue_per_user - control_revenue_per_user,
        2
    ) AS revenue_per_user_lift,

    ROUND(
        100.0 * (treatment_revenue_per_user - control_revenue_per_user) / NULLIF(control_revenue_per_user, 0),
        2
    ) AS revenue_per_user_uplift_pct,

    CASE
        WHEN treatment_conversion_rate_pct > control_conversion_rate_pct
             AND treatment_revenue_per_user > control_revenue_per_user
            THEN 'Treatment Wins'
        WHEN treatment_conversion_rate_pct < control_conversion_rate_pct
             AND treatment_revenue_per_user < control_revenue_per_user
            THEN 'Control Wins'
        ELSE 'Mixed Result'
    END AS experiment_result

FROM pivoted
ORDER BY revenue_per_user_uplift_pct DESC;
"""

con.execute(query)

print("bi_experiment_uplift_summary view created")

bi_experiment_uplift_summary view created


In [60]:
con.execute("""
SELECT *
FROM bi_experiment_uplift_summary
ORDER BY revenue_per_user_uplift_pct DESC
""").df()

,experiment_id,experiment_name,experiment_goal,control_users,treatment_users,control_ctr_pct,treatment_ctr_pct,ctr_lift_pp,ctr_uplift_pct,control_conversion_rate_pct,...,conversion_lift_pp,conversion_uplift_pct,control_revenue,treatment_revenue,incremental_revenue,control_revenue_per_user,treatment_revenue_per_user,revenue_per_user_lift,revenue_per_user_uplift_pct,experiment_result
0,EXP006,UA Rewards Early Access Personalization,increase_loyalty_member_retention,16672,16661,12.26,17.20,4.94,40.29,5.05,...,6.14,121.58,151909.44,331807.16,179897.72,9.11,19.92,10.81,118.66,Treatment Wins
1,EXP012,Loyalty Tier Upgrade Nudge Test,increase_loyalty_upgrade_rate,16686,16647,11.91,16.92,5.01,42.07,4.82,...,4.83,100.21,138098.85,291457.29,153358.44,8.28,17.51,9.23,111.47,Treatment Wins
2,EXP008,Discount Banner Personalization Test,reduce_discount_hunter_churn,16833,16500,12.49,18.92,6.43,51.48,4.77,...,4.83,101.26,127474.03,256436.51,128962.48,7.57,15.54,7.97,105.28,Treatment Wins
3,EXP003,Sport-Based Recommendation Engine v1,increase_recommendation_ctr,16551,16782,12.44,16.36,3.92,31.51,5.13,...,4.63,90.25,131506.50,258063.00,126556.50,7.95,15.38,7.43,93.46,Treatment Wins
4,EXP009,Holiday Season Recommendation Engine v2,increase_holiday_revenue,16474,16859,12.92,16.26,3.34,25.85,4.95,...,4.49,90.71,129937.47,255375.91,125438.44,7.89,15.15,7.26,92.02,Treatment Wins
5,EXP004,New Homepage Layout vs Old Layout,improve_homepage_conversion,16631,16702,11.90,19.18,7.28,61.18,5.27,...,4.09,77.61,137176.71,252984.37,115807.66,8.25,15.15,6.90,83.64,Treatment Wins
6,EXP007,Project Rock Collection Targeted Push,increase_recommendation_revenue,16653,16680,12.17,16.10,3.93,32.29,4.72,...,3.61,76.48,129570.38,229496.43,99926.05,7.78,13.76,5.98,76.86,Treatment Wins
7,EXP011,App vs Web Conversion Optimization,increase_app_conversion_rate,16668,16665,11.92,17.70,5.78,48.49,5.09,...,3.74,73.48,133063.58,233856.67,100793.09,7.98,14.03,6.05,75.81,Treatment Wins
8,EXP010,Sport Affinity Email Personalization,improve_email_click_through_rate,16638,16695,11.73,14.68,2.95,25.15,4.88,...,2.83,57.99,125534.61,204541.20,79006.59,7.55,12.25,4.70,62.25,Treatment Wins
9,EXP002,UA Rewards Loyalty Banner Test,increase_loyalty_member_conversion,16614,16719,11.92,15.52,3.60,30.20,4.82,...,2.36,48.96,138473.38,210948.43,72475.05,8.33,12.62,4.29,51.50,Treatment Wins


# **A/B Test Significance Check in SQL**

In [61]:
query = """
CREATE OR REPLACE VIEW bi_experiment_significance_summary AS
WITH base AS (
    SELECT
        experiment_id,
        experiment_name,
        experiment_goal,
        variant,
        COUNT(*) AS users,
        SUM(converted) AS conversions,
        1.0 * SUM(converted) / COUNT(*) AS conversion_rate
    FROM experiments_clean
    GROUP BY experiment_id, experiment_name, experiment_goal, variant
),

pivoted AS (
    SELECT
        experiment_id,
        experiment_name,
        experiment_goal,

        MAX(CASE WHEN variant = 'control' THEN users END) AS control_users,
        MAX(CASE WHEN variant = 'treatment' THEN users END) AS treatment_users,

        MAX(CASE WHEN variant = 'control' THEN conversions END) AS control_conversions,
        MAX(CASE WHEN variant = 'treatment' THEN conversions END) AS treatment_conversions,

        MAX(CASE WHEN variant = 'control' THEN conversion_rate END) AS control_conversion_rate,
        MAX(CASE WHEN variant = 'treatment' THEN conversion_rate END) AS treatment_conversion_rate
    FROM base
    GROUP BY experiment_id, experiment_name, experiment_goal
),

stats AS (
    SELECT
        *,

        1.0 * (control_conversions + treatment_conversions)
        / (control_users + treatment_users) AS pooled_conversion_rate

    FROM pivoted
),

z_calc AS (
    SELECT
        *,

        SQRT(
            pooled_conversion_rate
            * (1 - pooled_conversion_rate)
            * (1.0 / control_users + 1.0 / treatment_users)
        ) AS standard_error

    FROM stats
)

SELECT
    experiment_id,
    experiment_name,
    experiment_goal,

    control_users,
    treatment_users,
    control_conversions,
    treatment_conversions,

    ROUND(100 * control_conversion_rate, 2) AS control_conversion_rate_pct,
    ROUND(100 * treatment_conversion_rate, 2) AS treatment_conversion_rate_pct,

    ROUND(
        100 * (treatment_conversion_rate - control_conversion_rate),
        2
    ) AS conversion_lift_pp,

    ROUND(
        100 * (treatment_conversion_rate - control_conversion_rate)
        / NULLIF(control_conversion_rate, 0),
        2
    ) AS conversion_uplift_pct,

    ROUND(
        (treatment_conversion_rate - control_conversion_rate)
        / NULLIF(standard_error, 0),
        2
    ) AS z_score,

    CASE
        WHEN ABS(
            (treatment_conversion_rate - control_conversion_rate)
            / NULLIF(standard_error, 0)
        ) >= 1.96
            THEN 'Statistically Significant'
        ELSE 'Not Significant'
    END AS significance_result,

    CASE
        WHEN treatment_conversion_rate > control_conversion_rate
             AND ABS(
                (treatment_conversion_rate - control_conversion_rate)
                / NULLIF(standard_error, 0)
             ) >= 1.96
            THEN 'Roll Out Treatment'
        WHEN treatment_conversion_rate > control_conversion_rate
            THEN 'Promising, Need More Data'
        ELSE 'Do Not Roll Out'
    END AS recommendation

FROM z_calc
ORDER BY z_score DESC;
"""

con.execute(query)

print("bi_experiment_significance_summary view created")

bi_experiment_significance_summary view created


In [62]:
con.execute("""
SELECT *
FROM bi_experiment_significance_summary
ORDER BY z_score DESC
""").df()

,experiment_id,experiment_name,experiment_goal,control_users,treatment_users,control_conversions,treatment_conversions,control_conversion_rate_pct,treatment_conversion_rate_pct,conversion_lift_pp,conversion_uplift_pct,z_score,significance_result,recommendation
0,EXP006,UA Rewards Early Access Personalization,increase_loyalty_member_retention,16672,16661,842.0,1865.0,5.05,11.19,6.14,121.64,20.53,Statistically Significant,Roll Out Treatment
1,EXP008,Discount Banner Personalization Test,reduce_discount_hunter_churn,16833,16500,803.0,1584.0,4.77,9.60,4.83,101.24,17.10,Statistically Significant,Roll Out Treatment
2,EXP012,Loyalty Tier Upgrade Nudge Test,increase_loyalty_upgrade_rate,16686,16647,804.0,1606.0,4.82,9.65,4.83,100.22,17.02,Statistically Significant,Roll Out Treatment
3,EXP003,Sport-Based Recommendation Engine v1,increase_recommendation_ctr,16551,16782,849.0,1638.0,5.13,9.76,4.63,90.28,16.09,Statistically Significant,Roll Out Treatment
4,EXP009,Holiday Season Recommendation Engine v2,increase_holiday_revenue,16474,16859,816.0,1591.0,4.95,9.44,4.48,90.52,15.81,Statistically Significant,Roll Out Treatment
5,EXP004,New Homepage Layout vs Old Layout,improve_homepage_conversion,16631,16702,877.0,1564.0,5.27,9.36,4.09,77.58,14.33,Statistically Significant,Roll Out Treatment
6,EXP011,App vs Web Conversion Optimization,increase_app_conversion_rate,16668,16665,849.0,1472.0,5.09,8.83,3.74,73.41,13.41,Statistically Significant,Roll Out Treatment
7,EXP007,Project Rock Collection Targeted Push,increase_recommendation_revenue,16653,16680,786.0,1390.0,4.72,8.33,3.61,76.56,13.35,Statistically Significant,Roll Out Treatment
8,EXP001,Personalized vs Generic Homepage Recommendations,increase_recommendation_ctr,16577,16755,847.0,1383.0,5.11,8.25,3.14,61.55,11.49,Statistically Significant,Roll Out Treatment
9,EXP010,Sport Affinity Email Personalization,improve_email_click_through_rate,16638,16695,812.0,1287.0,4.88,7.71,2.83,57.96,10.63,Statistically Significant,Roll Out Treatment


# **Final Executive Summary KPI View**


In [63]:
query = """
CREATE OR REPLACE VIEW bi_executive_summary_kpis AS
WITH customer_kpis AS (
    SELECT
        COUNT(*) AS total_customers,
        SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
        SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,

        ROUND(
            100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS churn_rate_pct,

        ROUND(
            100.0 * SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS retention_rate_pct,

        ROUND(
            100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS loyalty_member_pct,

        ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value
    FROM customers_clean
),

revenue_kpis AS (
    SELECT
        ROUND(SUM(CASE WHEN status = 'completed' THEN total_amount ELSE 0 END), 2) AS completed_revenue,
        SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,

        ROUND(
            AVG(CASE WHEN status = 'completed' THEN total_amount END),
            2
        ) AS avg_completed_transaction_value,

        ROUND(
            100.0 * SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS completion_rate_pct
    FROM transactions_clean
),

repeat_kpis AS (
    WITH customer_purchase_months AS (
        SELECT
            customer_id,
            COUNT(DISTINCT DATE_TRUNC('month', transaction_date)) AS purchase_months
        FROM transactions_clean
        WHERE status = 'completed'
        GROUP BY customer_id
    )

    SELECT
        COUNT(*) AS purchasing_customers,

        ROUND(
            100.0 * SUM(CASE WHEN purchase_months > 1 THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS repeat_purchase_month_rate_pct
    FROM customer_purchase_months
),

recommendation_kpis AS (
    SELECT
        COUNT(*) AS recommendation_impressions,
        SUM(clicked) AS recommendation_clicks,
        SUM(purchased) AS recommendation_purchases,

        ROUND(
            100.0 * SUM(clicked) / COUNT(*),
            2
        ) AS recommendation_ctr_pct,

        ROUND(
            100.0 * SUM(purchased) / NULLIF(SUM(clicked), 0),
            2
        ) AS recommendation_cvr_after_click_pct,

        ROUND(SUM(revenue), 2) AS recommendation_revenue
    FROM recommendation_events_clean
),

experiment_kpis AS (
    SELECT
        COUNT(*) AS total_experiments,

        SUM(CASE WHEN experiment_result = 'Treatment Wins' THEN 1 ELSE 0 END) AS treatment_wins,

        ROUND(
            AVG(revenue_per_user_uplift_pct),
            2
        ) AS avg_revenue_per_user_uplift_pct,

        ROUND(
            SUM(incremental_revenue),
            2
        ) AS total_incremental_revenue
    FROM bi_experiment_uplift_summary
),

significance_kpis AS (
    SELECT
        SUM(CASE WHEN significance_result = 'Statistically Significant' THEN 1 ELSE 0 END) AS significant_experiments,

        SUM(CASE WHEN recommendation = 'Roll Out Treatment' THEN 1 ELSE 0 END) AS rollout_recommendations
    FROM bi_experiment_significance_summary
)

SELECT
    ck.total_customers,
    ck.active_customers,
    ck.churned_customers,
    ck.churn_rate_pct,
    ck.retention_rate_pct,
    ck.loyalty_member_pct,
    ck.avg_lifetime_value,

    rk.completed_revenue,
    rk.completed_transactions,
    rk.avg_completed_transaction_value,
    rk.completion_rate_pct,

    rep.purchasing_customers,
    rep.repeat_purchase_month_rate_pct,

    rec.recommendation_impressions,
    rec.recommendation_ctr_pct,
    rec.recommendation_cvr_after_click_pct,
    rec.recommendation_revenue,

    exp.total_experiments,
    exp.treatment_wins,
    sig.significant_experiments,
    sig.rollout_recommendations,
    exp.avg_revenue_per_user_uplift_pct,
    exp.total_incremental_revenue

FROM customer_kpis ck
CROSS JOIN revenue_kpis rk
CROSS JOIN repeat_kpis rep
CROSS JOIN recommendation_kpis rec
CROSS JOIN experiment_kpis exp
CROSS JOIN significance_kpis sig;
"""

con.execute(query)

print("bi_executive_summary_kpis view created")

bi_executive_summary_kpis view created


In [64]:
con.execute("SELECT * FROM bi_executive_summary_kpis").df()

,total_customers,active_customers,churned_customers,churn_rate_pct,retention_rate_pct,loyalty_member_pct,avg_lifetime_value,completed_revenue,completed_transactions,avg_completed_transaction_value,...,recommendation_impressions,recommendation_ctr_pct,recommendation_cvr_after_click_pct,recommendation_revenue,total_experiments,treatment_wins,significant_experiments,rollout_recommendations,avg_revenue_per_user_uplift_pct,total_incremental_revenue
0,200000,162869.0,37131.0,18.57,81.43,54.51,2219.95,1.901442e+08,1875600.0,101.38,...,600000,15.65,8.1,520395.7,12,12.0,12.0,12.0,79.97,1315914.58


# **Export all BI-ready CSV files**

In [65]:
output_path = "/kaggle/working/"

views_to_export = [
    "bi_executive_summary_kpis",
    "bi_monthly_revenue_kpis",
    "bi_segment_churn_revenue",
    "bi_rfm_customer_segments",
    "bi_rfm_segment_summary",
    "bi_cohort_retention_summary",
    "bi_cohort_milestone_summary",
    "bi_recommendation_model_performance",
    "bi_recommendation_segment_performance",
    "bi_personalized_vs_generic_uplift",
    "bi_experiment_ab_summary",
    "bi_experiment_uplift_summary",
    "bi_experiment_significance_summary"
]

for view in views_to_export:
    df = con.execute(f"SELECT * FROM {view}").df()
    file_path = output_path + view + ".csv"
    df.to_csv(file_path, index=False)
    print(f"Exported: {file_path} | Rows: {len(df)}")

print("All BI-ready CSV files exported successfully.")

Exported: /kaggle/working/bi_executive_summary_kpis.csv | Rows: 1
Exported: /kaggle/working/bi_monthly_revenue_kpis.csv | Rows: 62
Exported: /kaggle/working/bi_segment_churn_revenue.csv | Rows: 6
Exported: /kaggle/working/bi_rfm_customer_segments.csv | Rows: 193854
Exported: /kaggle/working/bi_rfm_segment_summary.csv | Rows: 8
Exported: /kaggle/working/bi_cohort_retention_summary.csv | Rows: 1953
Exported: /kaggle/working/bi_cohort_milestone_summary.csv | Rows: 62
Exported: /kaggle/working/bi_recommendation_model_performance.csv | Rows: 4
Exported: /kaggle/working/bi_recommendation_segment_performance.csv | Rows: 24
Exported: /kaggle/working/bi_personalized_vs_generic_uplift.csv | Rows: 6
Exported: /kaggle/working/bi_experiment_ab_summary.csv | Rows: 24
Exported: /kaggle/working/bi_experiment_uplift_summary.csv | Rows: 12
Exported: /kaggle/working/bi_experiment_significance_summary.csv | Rows: 12
All BI-ready CSV files exported successfully.


# **Export all BI-ready CSV files and zip them**

In [66]:
import os
import zipfile

output_path = "/kaggle/working/"
zip_file_path = "/kaggle/working/under_armour_project2_bi_ready_files.zip"

views_to_export = [
    "bi_executive_summary_kpis",
    "bi_monthly_revenue_kpis",
    "bi_segment_churn_revenue",
    "bi_rfm_customer_segments",
    "bi_rfm_segment_summary",
    "bi_cohort_retention_summary",
    "bi_cohort_milestone_summary",
    "bi_recommendation_model_performance",
    "bi_recommendation_segment_performance",
    "bi_personalized_vs_generic_uplift",
    "bi_experiment_ab_summary",
    "bi_experiment_uplift_summary",
    "bi_experiment_significance_summary"
]

csv_files = []

for view in views_to_export:
    df = con.execute(f"SELECT * FROM {view}").df()
    file_path = os.path.join(output_path, view + ".csv")
    df.to_csv(file_path, index=False)
    csv_files.append(file_path)
    print(f"Exported: {file_path} | Rows: {len(df)}")

with zipfile.ZipFile(zip_file_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in csv_files:
        zipf.write(file_path, arcname=os.path.basename(file_path))

print("All BI-ready CSV files exported and zipped successfully.")
print(f"Zip file saved at: {zip_file_path}")

Exported: /kaggle/working/bi_executive_summary_kpis.csv | Rows: 1
Exported: /kaggle/working/bi_monthly_revenue_kpis.csv | Rows: 62
Exported: /kaggle/working/bi_segment_churn_revenue.csv | Rows: 6
Exported: /kaggle/working/bi_rfm_customer_segments.csv | Rows: 193854
Exported: /kaggle/working/bi_rfm_segment_summary.csv | Rows: 8
Exported: /kaggle/working/bi_cohort_retention_summary.csv | Rows: 1953
Exported: /kaggle/working/bi_cohort_milestone_summary.csv | Rows: 62
Exported: /kaggle/working/bi_recommendation_model_performance.csv | Rows: 4
Exported: /kaggle/working/bi_recommendation_segment_performance.csv | Rows: 24
Exported: /kaggle/working/bi_personalized_vs_generic_uplift.csv | Rows: 6
Exported: /kaggle/working/bi_experiment_ab_summary.csv | Rows: 24
Exported: /kaggle/working/bi_experiment_uplift_summary.csv | Rows: 12
Exported: /kaggle/working/bi_experiment_significance_summary.csv | Rows: 12
All BI-ready CSV files exported and zipped successfully.
Zip file saved at: /kaggle/workin

# **Creating New Directory to save all SQL views**

In [7]:
import os
import shutil

sql_folder = "/kaggle/working/sql"
os.makedirs(sql_folder, exist_ok=True)

sql_files = {}

print("Ready. sql_files dictionary created.")

Ready. sql_files dictionary created.


# **Created 01_data_validation.sql**

In [8]:
sql_files["01_data_validation.sql"] = """
-- 01_data_validation.sql
-- Purpose: Validate row counts, duplicate keys, relationship integrity, and date ranges.

-- 1. Row count validation
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'transactions', COUNT(*) FROM transactions
UNION ALL
SELECT 'sessions', COUNT(*) FROM sessions
UNION ALL
SELECT 'reviews', COUNT(*) FROM reviews
UNION ALL
SELECT 'experiments', COUNT(*) FROM experiments
UNION ALL
SELECT 'recommendation_events', COUNT(*) FROM recommendation_events;


-- 2. Duplicate primary key check
SELECT 'customers' AS table_name, COUNT(*) AS total_rows, COUNT(DISTINCT customer_id) AS unique_ids,
       COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_ids
FROM customers
UNION ALL
SELECT 'products', COUNT(*), COUNT(DISTINCT product_id),
       COUNT(*) - COUNT(DISTINCT product_id)
FROM products
UNION ALL
SELECT 'transactions', COUNT(*), COUNT(DISTINCT transaction_id),
       COUNT(*) - COUNT(DISTINCT transaction_id)
FROM transactions
UNION ALL
SELECT 'sessions', COUNT(*), COUNT(DISTINCT session_id),
       COUNT(*) - COUNT(DISTINCT session_id)
FROM sessions
UNION ALL
SELECT 'reviews', COUNT(*), COUNT(DISTINCT review_id),
       COUNT(*) - COUNT(DISTINCT review_id)
FROM reviews
UNION ALL
SELECT 'experiments',
       COUNT(*),
       COUNT(DISTINCT experiment_id || '-' || customer_id || '-' || COALESCE(session_id, 'NO_SESSION') || '-' || exposure_date),
       COUNT(*) - COUNT(DISTINCT experiment_id || '-' || customer_id || '-' || COALESCE(session_id, 'NO_SESSION') || '-' || exposure_date)
FROM experiments
UNION ALL
SELECT 'recommendation_events', COUNT(*), COUNT(DISTINCT event_id),
       COUNT(*) - COUNT(DISTINCT event_id)
FROM recommendation_events;


-- 3. Find duplicate experiment exposure rows
SELECT
    experiment_id,
    customer_id,
    COALESCE(session_id, 'NO_SESSION') AS session_id_clean,
    exposure_date,
    COUNT(*) AS duplicate_count
FROM experiments
GROUP BY experiment_id, customer_id, COALESCE(session_id, 'NO_SESSION'), exposure_date
HAVING COUNT(*) > 1;


-- 4. Relationship integrity check
SELECT 'transactions → customers' AS relationship, COUNT(*) AS missing_rows
FROM transactions t
LEFT JOIN customers c ON t.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 'transactions → products', COUNT(*)
FROM transactions t
LEFT JOIN products p ON t.product_id = p.product_id
WHERE p.product_id IS NULL

UNION ALL

SELECT 'sessions → customers', COUNT(*)
FROM sessions s
LEFT JOIN customers c ON s.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 'reviews → customers', COUNT(*)
FROM reviews r
LEFT JOIN customers c ON r.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 'reviews → products', COUNT(*)
FROM reviews r
LEFT JOIN products p ON r.product_id = p.product_id
WHERE p.product_id IS NULL

UNION ALL

SELECT 'experiments → customers', COUNT(*)
FROM experiments e
LEFT JOIN customers c ON e.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 'recommendation_events → customers', COUNT(*)
FROM recommendation_events re
LEFT JOIN customers c ON re.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 'recommendation_events → products', COUNT(*)
FROM recommendation_events re
LEFT JOIN products p ON re.product_id = p.product_id
WHERE p.product_id IS NULL;


-- 5. Date range validation
SELECT 'customers.signup_date' AS date_field,
       MIN(CAST(signup_date AS DATE)) AS min_date,
       MAX(CAST(signup_date AS DATE)) AS max_date
FROM customers

UNION ALL

SELECT 'transactions.transaction_date',
       MIN(CAST(transaction_date AS DATE)),
       MAX(CAST(transaction_date AS DATE))
FROM transactions

UNION ALL

SELECT 'sessions.session_date',
       MIN(CAST(session_date AS DATE)),
       MAX(CAST(session_date AS DATE))
FROM sessions

UNION ALL

SELECT 'reviews.review_date',
       MIN(CAST(review_date AS DATE)),
       MAX(CAST(review_date AS DATE))
FROM reviews

UNION ALL

SELECT 'experiments.exposure_date',
       MIN(CAST(exposure_date AS DATE)),
       MAX(CAST(exposure_date AS DATE))
FROM experiments

UNION ALL

SELECT 'recommendation_events.event_date',
       MIN(CAST(event_date AS DATE)),
       MAX(CAST(event_date AS DATE))
FROM recommendation_events;
"""

# **Created 02_clean_views.sql**

In [9]:
sql_files["02_clean_views.sql"] = """
-- 02_clean_views.sql
-- Purpose: Create clean views for analysis.

CREATE OR REPLACE VIEW experiments_clean AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY experiment_id, customer_id, COALESCE(session_id, 'NO_SESSION'), exposure_date
            ORDER BY exposure_date
        ) AS rn
    FROM experiments
)
WHERE rn = 1;


CREATE OR REPLACE VIEW customers_clean AS
SELECT
    customer_id,
    CAST(signup_date AS DATE) AS signup_date,
    age,
    COALESCE(gender, 'Unknown') AS gender,
    country,
    segment,
    is_churned,
    COALESCE(lifetime_value, 0) AS lifetime_value,
    is_loyalty_member,
    ua_rewards_points,
    email_opt_in,
    has_app,
    COALESCE(preferred_sport, 'Unknown') AS preferred_sport
FROM customers;


CREATE OR REPLACE VIEW products_clean AS
SELECT
    product_id,
    product_name,
    category,
    brand,
    gender_target,
    price,
    avg_rating,
    num_ratings,
    stock_quantity,
    discount_pct,
    is_featured,
    is_new_arrival,
    weight_kg
FROM products;


CREATE OR REPLACE VIEW transactions_clean AS
SELECT
    transaction_id,
    customer_id,
    product_id,
    CAST(transaction_date AS TIMESTAMP) AS transaction_date,
    quantity,
    unit_price,
    total_amount,
    discount_applied,
    loyalty_discount,
    status,
    payment_method,
    shipping_cost
FROM transactions;


CREATE OR REPLACE VIEW sessions_clean AS
SELECT
    session_id,
    customer_id,
    CAST(session_date AS TIMESTAMP) AS session_date,
    device,
    COALESCE(channel, 'unknown') AS channel,
    duration_seconds,
    pages_viewed,
    converted,
    bounced,
    cart_additions,
    is_loyalty_session
FROM sessions;


CREATE OR REPLACE VIEW reviews_clean AS
SELECT
    review_id,
    customer_id,
    product_id,
    CAST(review_date AS DATE) AS review_date,
    rating,
    review_text,
    helpful_votes,
    verified_purchase,
    review_source
FROM reviews;


CREATE OR REPLACE VIEW recommendation_events_clean AS
SELECT
    event_id,
    customer_id,
    session_id,
    product_id,
    recommendation_model,
    CAST(event_date AS TIMESTAMP) AS event_date,
    impression,
    clicked,
    purchased,
    COALESCE(revenue, 0) AS revenue
FROM recommendation_events;
"""

# **Created 03_executive_kpis.sql**

In [10]:
sql_files["03_executive_kpis.sql"] = """
-- 03_executive_kpis.sql
-- Purpose: Executive customer and business KPI analysis.

SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,

    ROUND(100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS retention_rate_pct,

    SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) AS loyalty_members,
    ROUND(100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loyalty_member_pct,

    SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) AS app_users,
    ROUND(100.0 * SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS app_user_pct,

    SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) AS email_opt_in_users,
    ROUND(100.0 * SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS email_opt_in_pct,

    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value
FROM customers_clean;


CREATE OR REPLACE VIEW bi_executive_summary_kpis AS
WITH customer_kpis AS (
    SELECT
        COUNT(*) AS total_customers,
        SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
        SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
        ROUND(100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
        ROUND(100.0 * SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS retention_rate_pct,
        ROUND(100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loyalty_member_pct,
        ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value
    FROM customers_clean
),

revenue_kpis AS (
    SELECT
        ROUND(SUM(CASE WHEN status = 'completed' THEN total_amount ELSE 0 END), 2) AS completed_revenue,
        SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,
        ROUND(AVG(CASE WHEN status = 'completed' THEN total_amount END), 2) AS avg_completed_transaction_value,
        ROUND(100.0 * SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS completion_rate_pct
    FROM transactions_clean
),

repeat_kpis AS (
    WITH customer_purchase_months AS (
        SELECT
            customer_id,
            COUNT(DISTINCT DATE_TRUNC('month', transaction_date)) AS purchase_months
        FROM transactions_clean
        WHERE status = 'completed'
        GROUP BY customer_id
    )
    SELECT
        COUNT(*) AS purchasing_customers,
        ROUND(100.0 * SUM(CASE WHEN purchase_months > 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS repeat_purchase_month_rate_pct
    FROM customer_purchase_months
),

recommendation_kpis AS (
    SELECT
        COUNT(*) AS recommendation_impressions,
        ROUND(100.0 * SUM(clicked) / COUNT(*), 2) AS recommendation_ctr_pct,
        ROUND(100.0 * SUM(purchased) / NULLIF(SUM(clicked), 0), 2) AS recommendation_cvr_after_click_pct,
        ROUND(SUM(revenue), 2) AS recommendation_revenue
    FROM recommendation_events_clean
),

experiment_kpis AS (
    SELECT
        COUNT(*) AS total_experiments,
        SUM(CASE WHEN experiment_result = 'Treatment Wins' THEN 1 ELSE 0 END) AS treatment_wins,
        ROUND(AVG(revenue_per_user_uplift_pct), 2) AS avg_revenue_per_user_uplift_pct,
        ROUND(SUM(incremental_revenue), 2) AS total_incremental_revenue
    FROM bi_experiment_uplift_summary
),

significance_kpis AS (
    SELECT
        SUM(CASE WHEN significance_result = 'Statistically Significant' THEN 1 ELSE 0 END) AS significant_experiments,
        SUM(CASE WHEN recommendation = 'Roll Out Treatment' THEN 1 ELSE 0 END) AS rollout_recommendations
    FROM bi_experiment_significance_summary
)

SELECT
    ck.total_customers,
    ck.active_customers,
    ck.churned_customers,
    ck.churn_rate_pct,
    ck.retention_rate_pct,
    ck.loyalty_member_pct,
    ck.avg_lifetime_value,
    rk.completed_revenue,
    rk.completed_transactions,
    rk.avg_completed_transaction_value,
    rk.completion_rate_pct,
    rep.purchasing_customers,
    rep.repeat_purchase_month_rate_pct,
    rec.recommendation_impressions,
    rec.recommendation_ctr_pct,
    rec.recommendation_cvr_after_click_pct,
    rec.recommendation_revenue,
    exp.total_experiments,
    exp.treatment_wins,
    sig.significant_experiments,
    sig.rollout_recommendations,
    exp.avg_revenue_per_user_uplift_pct,
    exp.total_incremental_revenue
FROM customer_kpis ck
CROSS JOIN revenue_kpis rk
CROSS JOIN repeat_kpis rep
CROSS JOIN recommendation_kpis rec
CROSS JOIN experiment_kpis exp
CROSS JOIN significance_kpis sig;
"""

# **Created 04_churn_analysis.sql**

In [11]:
sql_files["04_churn_analysis.sql"] = """
-- 04_churn_analysis.sql
-- Purpose: Analyze churn by segment, loyalty, app, email, and country.

-- Churn by customer segment
SELECT
    segment,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
    ROUND(100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,
    SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) AS loyalty_members,
    ROUND(100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loyalty_member_pct
FROM customers_clean
GROUP BY segment
ORDER BY churn_rate_pct DESC;


-- Churn by loyalty membership
SELECT
    CASE WHEN is_loyalty_member = 1 THEN 'Loyalty Member' ELSE 'Non-Loyalty Member' END AS loyalty_status,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
    ROUND(100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,
    ROUND(AVG(ua_rewards_points), 2) AS avg_rewards_points,
    ROUND(100.0 * SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS app_user_pct,
    ROUND(100.0 * SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS email_opt_in_pct
FROM customers_clean
GROUP BY loyalty_status
ORDER BY churn_rate_pct DESC;


-- Churn by app usage
SELECT
    CASE WHEN has_app = 1 THEN 'App User' ELSE 'Non-App User' END AS app_status,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
    ROUND(100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,
    ROUND(100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loyalty_member_pct,
    ROUND(100.0 * SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS email_opt_in_pct
FROM customers_clean
GROUP BY app_status
ORDER BY churn_rate_pct DESC;


-- Churn by email opt-in
SELECT
    CASE WHEN email_opt_in = 1 THEN 'Email Opt-in' ELSE 'No Email Opt-in' END AS email_status,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
    ROUND(100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,
    ROUND(100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loyalty_member_pct,
    ROUND(100.0 * SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS app_user_pct
FROM customers_clean
GROUP BY email_status
ORDER BY churn_rate_pct DESC;


-- Churn by country
SELECT
    country,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
    ROUND(100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,
    ROUND(100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loyalty_member_pct,
    ROUND(100.0 * SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS app_user_pct,
    ROUND(100.0 * SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS email_opt_in_pct
FROM customers_clean
GROUP BY country
ORDER BY churn_rate_pct DESC;
"""

# **Created 05_revenue_analysis.sql**

In [12]:
sql_files["05_revenue_analysis.sql"] = """
-- 05_revenue_analysis.sql
-- Purpose: Analyze revenue health, order status, revenue leakage, and monthly revenue KPIs.

-- Revenue overview
SELECT
    COUNT(*) AS total_transaction_rows,
    SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,
    SUM(CASE WHEN status = 'returned' THEN 1 ELSE 0 END) AS returned_transactions,
    SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_transactions,
    SUM(CASE WHEN status = 'pending' THEN 1 ELSE 0 END) AS pending_transactions,
    ROUND(100.0 * SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS completed_transaction_pct,
    ROUND(SUM(CASE WHEN status = 'completed' THEN total_amount ELSE 0 END), 2) AS completed_revenue,
    ROUND(AVG(CASE WHEN status = 'completed' THEN total_amount END), 2) AS avg_completed_transaction_value,
    SUM(CASE WHEN status = 'completed' THEN quantity ELSE 0 END) AS completed_units_sold,
    ROUND(SUM(CASE WHEN status = 'returned' THEN total_amount ELSE 0 END), 2) AS returned_revenue_value,
    ROUND(SUM(CASE WHEN status = 'cancelled' THEN total_amount ELSE 0 END), 2) AS cancelled_revenue_value
FROM transactions_clean;


-- Revenue and order status by customer segment
SELECT
    c.segment,
    COUNT(*) AS total_transaction_rows,
    SUM(CASE WHEN t.status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,
    SUM(CASE WHEN t.status = 'returned' THEN 1 ELSE 0 END) AS returned_transactions,
    SUM(CASE WHEN t.status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_transactions,
    ROUND(100.0 * SUM(CASE WHEN t.status = 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS completion_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN t.status = 'returned' THEN 1 ELSE 0 END) / COUNT(*), 2) AS return_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN t.status = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancellation_rate_pct,
    ROUND(SUM(CASE WHEN t.status = 'completed' THEN t.total_amount ELSE 0 END), 2) AS completed_revenue,
    ROUND(AVG(CASE WHEN t.status = 'completed' THEN t.total_amount END), 2) AS avg_completed_transaction_value,
    COUNT(DISTINCT t.customer_id) AS purchasing_customers,
    ROUND(SUM(CASE WHEN t.status = 'completed' THEN t.total_amount ELSE 0 END) / COUNT(DISTINCT t.customer_id), 2) AS revenue_per_purchasing_customer
FROM transactions_clean t
JOIN customers_clean c ON t.customer_id = c.customer_id
GROUP BY c.segment
ORDER BY completed_revenue DESC;


-- BI-ready monthly revenue KPI view
CREATE OR REPLACE VIEW bi_monthly_revenue_kpis AS
SELECT
    DATE_TRUNC('month', transaction_date) AS month,
    COUNT(*) AS total_transaction_rows,
    SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,
    SUM(CASE WHEN status = 'returned' THEN 1 ELSE 0 END) AS returned_transactions,
    SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_transactions,
    SUM(CASE WHEN status = 'pending' THEN 1 ELSE 0 END) AS pending_transactions,
    COUNT(DISTINCT CASE WHEN status = 'completed' THEN customer_id END) AS purchasing_customers,
    ROUND(SUM(CASE WHEN status = 'completed' THEN total_amount ELSE 0 END), 2) AS completed_revenue,
    ROUND(AVG(CASE WHEN status = 'completed' THEN total_amount END), 2) AS avg_completed_transaction_value,
    SUM(CASE WHEN status = 'completed' THEN quantity ELSE 0 END) AS completed_units_sold,
    ROUND(100.0 * SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS completion_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN status = 'returned' THEN 1 ELSE 0 END) / COUNT(*), 2) AS return_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancellation_rate_pct
FROM transactions_clean
GROUP BY DATE_TRUNC('month', transaction_date);


-- BI-ready segment churn and revenue view
CREATE OR REPLACE VIEW bi_segment_churn_revenue AS
WITH segment_customer_base AS (
    SELECT
        segment,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
        SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
        SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) AS loyalty_members,
        ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value
    FROM customers_clean
    GROUP BY segment
),

segment_revenue AS (
    SELECT
        c.segment,
        COUNT(DISTINCT t.customer_id) AS purchasing_customers,
        SUM(CASE WHEN t.status = 'completed' THEN 1 ELSE 0 END) AS completed_transactions,
        ROUND(SUM(CASE WHEN t.status = 'completed' THEN t.total_amount ELSE 0 END), 2) AS completed_revenue,
        ROUND(AVG(CASE WHEN t.status = 'completed' THEN t.total_amount END), 2) AS avg_completed_transaction_value
    FROM transactions_clean t
    JOIN customers_clean c ON t.customer_id = c.customer_id
    GROUP BY c.segment
)

SELECT
    cb.segment,
    cb.total_customers,
    cb.active_customers,
    cb.churned_customers,
    ROUND(100.0 * cb.churned_customers / cb.total_customers, 2) AS churn_rate_pct,
    ROUND(100.0 * cb.active_customers / cb.total_customers, 2) AS retention_rate_pct,
    cb.loyalty_members,
    ROUND(100.0 * cb.loyalty_members / cb.total_customers, 2) AS loyalty_member_pct,
    cb.avg_lifetime_value,
    COALESCE(sr.purchasing_customers, 0) AS purchasing_customers,
    COALESCE(sr.completed_transactions, 0) AS completed_transactions,
    COALESCE(sr.completed_revenue, 0) AS completed_revenue,
    COALESCE(sr.avg_completed_transaction_value, 0) AS avg_completed_transaction_value,
    ROUND(COALESCE(sr.completed_revenue, 0) / cb.total_customers, 2) AS revenue_per_customer
FROM segment_customer_base cb
LEFT JOIN segment_revenue sr ON cb.segment = sr.segment
ORDER BY churn_rate_pct DESC;
"""

# **Created 06_rfm_segmentation.sql**

In [13]:
sql_files["06_rfm_segmentation.sql"] = """
-- 06_rfm_segmentation.sql
-- Purpose: Create RFM customer segmentation and recommended retention actions.

CREATE OR REPLACE VIEW rfm_base AS
WITH max_date AS (
    SELECT MAX(CAST(transaction_date AS DATE)) AS analysis_date
    FROM transactions_clean
    WHERE status = 'completed'
)

SELECT
    t.customer_id,
    MIN(CAST(t.transaction_date AS DATE)) AS first_purchase_date,
    MAX(CAST(t.transaction_date AS DATE)) AS last_purchase_date,
    DATE_DIFF('day', MAX(CAST(t.transaction_date AS DATE)), (SELECT analysis_date FROM max_date)) AS recency_days,
    COUNT(DISTINCT t.transaction_id) AS frequency,
    ROUND(SUM(t.total_amount), 2) AS monetary_value,
    ROUND(AVG(t.total_amount), 2) AS avg_order_value
FROM transactions_clean t
WHERE t.status = 'completed'
GROUP BY t.customer_id;


CREATE OR REPLACE VIEW rfm_scored AS
SELECT
    customer_id,
    first_purchase_date,
    last_purchase_date,
    recency_days,
    frequency,
    monetary_value,
    avg_order_value,
    NTILE(5) OVER (ORDER BY recency_days DESC, customer_id) AS r_score,
    NTILE(5) OVER (ORDER BY frequency ASC, customer_id) AS f_score,
    NTILE(5) OVER (ORDER BY monetary_value ASC, customer_id) AS m_score
FROM rfm_base;


CREATE OR REPLACE VIEW rfm_segments AS
SELECT
    customer_id,
    first_purchase_date,
    last_purchase_date,
    recency_days,
    frequency,
    monetary_value,
    avg_order_value,
    r_score,
    f_score,
    m_score,
    CAST(r_score AS VARCHAR) || CAST(f_score AS VARCHAR) || CAST(m_score AS VARCHAR) AS rfm_score,

    CASE
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'VIP Customers'
        WHEN r_score >= 4 AND f_score >= 3 THEN 'Loyal Customers'
        WHEN r_score >= 4 AND f_score <= 2 THEN 'New / Promising Customers'
        WHEN r_score <= 2 AND f_score >= 4 AND m_score >= 4 THEN 'At-Risk High-Value Customers'
        WHEN r_score <= 2 AND f_score >= 3 THEN 'At-Risk Loyal Customers'
        WHEN r_score <= 2 AND f_score <= 2 THEN 'Lost / Dormant Customers'
        WHEN m_score >= 4 AND f_score <= 2 THEN 'Big Spenders - Low Frequency'
        ELSE 'Regular Customers'
    END AS rfm_segment
FROM rfm_scored;


CREATE OR REPLACE VIEW bi_rfm_customer_segments AS
SELECT
    r.customer_id,
    c.segment AS original_customer_segment,
    r.rfm_segment,
    r.rfm_score,
    r.first_purchase_date,
    r.last_purchase_date,
    r.recency_days,
    r.frequency,
    r.monetary_value,
    r.avg_order_value,
    r.r_score,
    r.f_score,
    r.m_score,
    c.country,
    c.gender,
    c.age,
    c.is_churned,
    c.is_loyalty_member,
    c.ua_rewards_points,
    c.email_opt_in,
    c.has_app,
    c.preferred_sport,
    c.lifetime_value,

    CASE
        WHEN r.rfm_segment IN ('At-Risk High-Value Customers', 'At-Risk Loyal Customers') THEN 'Win Back'
        WHEN r.rfm_segment = 'VIP Customers' THEN 'Protect and Reward'
        WHEN r.rfm_segment = 'Lost / Dormant Customers' THEN 'Reactivation'
        WHEN r.rfm_segment = 'New / Promising Customers' THEN 'Nurture'
        WHEN r.rfm_segment = 'Big Spenders - Low Frequency' THEN 'Increase Frequency'
        ELSE 'Maintain Engagement'
    END AS recommended_action

FROM rfm_segments r
JOIN customers_clean c ON r.customer_id = c.customer_id;


CREATE OR REPLACE VIEW bi_rfm_segment_summary AS
SELECT
    rfm_segment,
    recommended_action,
    COUNT(*) AS customers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS customer_pct,
    ROUND(AVG(recency_days), 2) AS avg_recency_days,
    ROUND(AVG(frequency), 2) AS avg_frequency,
    ROUND(AVG(monetary_value), 2) AS avg_monetary_value,
    ROUND(AVG(avg_order_value), 2) AS avg_order_value,
    ROUND(SUM(monetary_value), 2) AS total_monetary_value,
    SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) AS active_customers,
    ROUND(100.0 * SUM(CASE WHEN is_churned = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN is_churned = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS retention_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN is_loyalty_member = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loyalty_member_pct,
    ROUND(100.0 * SUM(CASE WHEN has_app = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS app_user_pct,
    ROUND(100.0 * SUM(CASE WHEN email_opt_in = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS email_opt_in_pct
FROM bi_rfm_customer_segments
GROUP BY rfm_segment, recommended_action;
"""

# **Created 07_cohort_analysis.sql**

In [14]:
sql_files["07_cohort_analysis.sql"] = """
-- 07_cohort_analysis.sql
-- Purpose: Cohort retention and repeat behavior analysis.

CREATE OR REPLACE VIEW cohort_base AS
WITH first_purchase AS (
    SELECT
        customer_id,
        DATE_TRUNC('month', MIN(transaction_date)) AS cohort_month
    FROM transactions_clean
    WHERE status = 'completed'
    GROUP BY customer_id
),

customer_purchase_months AS (
    SELECT DISTINCT
        customer_id,
        DATE_TRUNC('month', transaction_date) AS purchase_month
    FROM transactions_clean
    WHERE status = 'completed'
)

SELECT
    fp.customer_id,
    fp.cohort_month,
    cpm.purchase_month,
    DATE_DIFF('month', fp.cohort_month, cpm.purchase_month) AS cohort_age_month
FROM first_purchase fp
JOIN customer_purchase_months cpm ON fp.customer_id = cpm.customer_id
WHERE cpm.purchase_month >= fp.cohort_month;


CREATE OR REPLACE VIEW bi_cohort_retention_summary AS
WITH cohort_counts AS (
    SELECT
        cohort_month,
        COUNT(DISTINCT customer_id) AS cohort_size
    FROM cohort_base
    WHERE cohort_age_month = 0
    GROUP BY cohort_month
),

retention_counts AS (
    SELECT
        cohort_month,
        cohort_age_month,
        COUNT(DISTINCT customer_id) AS retained_customers
    FROM cohort_base
    GROUP BY cohort_month, cohort_age_month
)

SELECT
    rc.cohort_month,
    rc.cohort_age_month,
    cc.cohort_size,
    rc.retained_customers,
    ROUND(100.0 * rc.retained_customers / cc.cohort_size, 2) AS retention_rate_pct
FROM retention_counts rc
JOIN cohort_counts cc ON rc.cohort_month = cc.cohort_month
ORDER BY rc.cohort_month, rc.cohort_age_month;


CREATE OR REPLACE VIEW bi_cohort_milestone_summary AS
SELECT
    cohort_month,
    MAX(CASE WHEN cohort_age_month = 0 THEN cohort_size END) AS cohort_size,
    MAX(CASE WHEN cohort_age_month = 1 THEN retention_rate_pct END) AS month_1_retention_pct,
    MAX(CASE WHEN cohort_age_month = 3 THEN retention_rate_pct END) AS month_3_retention_pct,
    MAX(CASE WHEN cohort_age_month = 6 THEN retention_rate_pct END) AS month_6_retention_pct,
    MAX(CASE WHEN cohort_age_month = 12 THEN retention_rate_pct END) AS month_12_retention_pct,
    MAX(CASE WHEN cohort_age_month = 1 THEN retained_customers END) AS month_1_retained_customers,
    MAX(CASE WHEN cohort_age_month = 3 THEN retained_customers END) AS month_3_retained_customers,
    MAX(CASE WHEN cohort_age_month = 6 THEN retained_customers END) AS month_6_retained_customers,
    MAX(CASE WHEN cohort_age_month = 12 THEN retained_customers END) AS month_12_retained_customers
FROM bi_cohort_retention_summary
GROUP BY cohort_month
ORDER BY cohort_month;
"""

# **Created 08_personalization_analysis.sql**

In [15]:
sql_files["08_personalization_analysis.sql"] = """
-- 08_personalization_analysis.sql
-- Purpose: Analyze recommendation model performance and personalized vs generic uplift.

CREATE OR REPLACE VIEW bi_recommendation_model_performance AS
SELECT
    recommendation_model,
    COUNT(*) AS impressions,
    SUM(clicked) AS clicks,
    SUM(purchased) AS purchases,
    ROUND(100.0 * SUM(clicked) / COUNT(*), 2) AS ctr_pct,
    ROUND(100.0 * SUM(purchased) / NULLIF(SUM(clicked), 0), 2) AS cvr_after_click_pct,
    ROUND(100.0 * SUM(purchased) / COUNT(*), 2) AS purchase_rate_per_impression_pct,
    ROUND(SUM(revenue), 2) AS recommendation_revenue,
    ROUND(SUM(revenue) / COUNT(*), 4) AS revenue_per_impression,
    ROUND(SUM(revenue) / NULLIF(SUM(clicked), 0), 2) AS revenue_per_click
FROM recommendation_events_clean
GROUP BY recommendation_model
ORDER BY recommendation_revenue DESC;


CREATE OR REPLACE VIEW bi_recommendation_segment_performance AS
SELECT
    c.segment,
    re.recommendation_model,
    COUNT(*) AS impressions,
    SUM(re.clicked) AS clicks,
    SUM(re.purchased) AS purchases,
    ROUND(100.0 * SUM(re.clicked) / COUNT(*), 2) AS ctr_pct,
    ROUND(100.0 * SUM(re.purchased) / NULLIF(SUM(re.clicked), 0), 2) AS cvr_after_click_pct,
    ROUND(100.0 * SUM(re.purchased) / COUNT(*), 2) AS purchase_rate_per_impression_pct,
    ROUND(SUM(re.revenue), 2) AS recommendation_revenue,
    ROUND(SUM(re.revenue) / COUNT(*), 4) AS revenue_per_impression
FROM recommendation_events_clean re
JOIN customers_clean c ON re.customer_id = c.customer_id
GROUP BY c.segment, re.recommendation_model;


CREATE OR REPLACE VIEW bi_personalized_vs_generic_uplift AS
WITH model_perf AS (
    SELECT
        c.segment,
        re.recommendation_model,
        COUNT(*) AS impressions,
        SUM(re.clicked) AS clicks,
        SUM(re.purchased) AS purchases,
        SUM(re.revenue) AS revenue,
        1.0 * SUM(re.clicked) / COUNT(*) AS ctr,
        1.0 * SUM(re.purchased) / COUNT(*) AS purchase_rate,
        1.0 * SUM(re.revenue) / COUNT(*) AS revenue_per_impression
    FROM recommendation_events_clean re
    JOIN customers_clean c ON re.customer_id = c.customer_id
    WHERE re.recommendation_model IN ('personalized', 'generic')
    GROUP BY c.segment, re.recommendation_model
),

pivoted AS (
    SELECT
        segment,
        MAX(CASE WHEN recommendation_model = 'personalized' THEN impressions END) AS personalized_impressions,
        MAX(CASE WHEN recommendation_model = 'generic' THEN impressions END) AS generic_impressions,
        MAX(CASE WHEN recommendation_model = 'personalized' THEN ctr END) AS personalized_ctr,
        MAX(CASE WHEN recommendation_model = 'generic' THEN ctr END) AS generic_ctr,
        MAX(CASE WHEN recommendation_model = 'personalized' THEN purchase_rate END) AS personalized_purchase_rate,
        MAX(CASE WHEN recommendation_model = 'generic' THEN purchase_rate END) AS generic_purchase_rate,
        MAX(CASE WHEN recommendation_model = 'personalized' THEN revenue_per_impression END) AS personalized_revenue_per_impression,
        MAX(CASE WHEN recommendation_model = 'generic' THEN revenue_per_impression END) AS generic_revenue_per_impression,
        MAX(CASE WHEN recommendation_model = 'personalized' THEN revenue END) AS personalized_revenue,
        MAX(CASE WHEN recommendation_model = 'generic' THEN revenue END) AS generic_revenue
    FROM model_perf
    GROUP BY segment
)

SELECT
    segment,
    personalized_impressions,
    generic_impressions,
    ROUND(100 * personalized_ctr, 2) AS personalized_ctr_pct,
    ROUND(100 * generic_ctr, 2) AS generic_ctr_pct,
    ROUND(100.0 * (personalized_ctr - generic_ctr) / NULLIF(generic_ctr, 0), 2) AS ctr_uplift_pct,
    ROUND(100 * personalized_purchase_rate, 2) AS personalized_purchase_rate_pct,
    ROUND(100 * generic_purchase_rate, 2) AS generic_purchase_rate_pct,
    ROUND(100.0 * (personalized_purchase_rate - generic_purchase_rate) / NULLIF(generic_purchase_rate, 0), 2) AS purchase_rate_uplift_pct,
    ROUND(personalized_revenue_per_impression, 4) AS personalized_revenue_per_impression,
    ROUND(generic_revenue_per_impression, 4) AS generic_revenue_per_impression,
    ROUND(100.0 * (personalized_revenue_per_impression - generic_revenue_per_impression) / NULLIF(generic_revenue_per_impression, 0), 2) AS revenue_per_impression_uplift_pct,
    ROUND(personalized_revenue, 2) AS personalized_revenue,
    ROUND(generic_revenue, 2) AS generic_revenue
FROM pivoted
ORDER BY revenue_per_impression_uplift_pct DESC;
"""

# **Created 09_ab_testing_analysis.sql**

In [22]:
sql_files["09_ab_testing_analysis.sql"] = """
-- 09_ab_testing_analysis.sql
-- Purpose: A/B testing, uplift calculation, and statistical significance.

CREATE OR REPLACE VIEW bi_experiment_ab_summary AS
SELECT
    experiment_id,
    experiment_name,
    experiment_goal,
    variant,
    COUNT(*) AS exposed_users,
    SUM(clicked_recommendation) AS clicks,
    SUM(converted) AS conversions,
    ROUND(100.0 * SUM(clicked_recommendation) / COUNT(*), 2) AS ctr_pct,
    ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct,
    ROUND(SUM(revenue), 2) AS revenue,
    ROUND(SUM(revenue) / COUNT(*), 2) AS revenue_per_exposed_user
FROM experiments_clean
GROUP BY experiment_id, experiment_name, experiment_goal, variant;


CREATE OR REPLACE VIEW bi_experiment_uplift_summary AS
WITH pivoted AS (
    SELECT
        experiment_id,
        experiment_name,
        experiment_goal,
        MAX(CASE WHEN variant = 'control' THEN exposed_users END) AS control_users,
        MAX(CASE WHEN variant = 'treatment' THEN exposed_users END) AS treatment_users,
        MAX(CASE WHEN variant = 'control' THEN ctr_pct END) AS control_ctr_pct,
        MAX(CASE WHEN variant = 'treatment' THEN ctr_pct END) AS treatment_ctr_pct,
        MAX(CASE WHEN variant = 'control' THEN conversion_rate_pct END) AS control_conversion_rate_pct,
        MAX(CASE WHEN variant = 'treatment' THEN conversion_rate_pct END) AS treatment_conversion_rate_pct,
        MAX(CASE WHEN variant = 'control' THEN revenue END) AS control_revenue,
        MAX(CASE WHEN variant = 'treatment' THEN revenue END) AS treatment_revenue,
        MAX(CASE WHEN variant = 'control' THEN revenue_per_exposed_user END) AS control_revenue_per_user,
        MAX(CASE WHEN variant = 'treatment' THEN revenue_per_exposed_user END) AS treatment_revenue_per_user
    FROM bi_experiment_ab_summary
    GROUP BY experiment_id, experiment_name, experiment_goal
)

SELECT
    experiment_id,
    experiment_name,
    experiment_goal,
    control_users,
    treatment_users,
    control_ctr_pct,
    treatment_ctr_pct,
    ROUND(treatment_ctr_pct - control_ctr_pct, 2) AS ctr_lift_pp,
    ROUND(100.0 * (treatment_ctr_pct - control_ctr_pct) / NULLIF(control_ctr_pct, 0), 2) AS ctr_uplift_pct,
    control_conversion_rate_pct,
    treatment_conversion_rate_pct,
    ROUND(treatment_conversion_rate_pct - control_conversion_rate_pct, 2) AS conversion_lift_pp,
    ROUND(100.0 * (treatment_conversion_rate_pct - control_conversion_rate_pct) / NULLIF(control_conversion_rate_pct, 0), 2) AS conversion_uplift_pct,
    control_revenue,
    treatment_revenue,
    ROUND(treatment_revenue - control_revenue, 2) AS incremental_revenue,
    control_revenue_per_user,
    treatment_revenue_per_user,
    ROUND(treatment_revenue_per_user - control_revenue_per_user, 2) AS revenue_per_user_lift,
    ROUND(100.0 * (treatment_revenue_per_user - control_revenue_per_user) / NULLIF(control_revenue_per_user, 0), 2) AS revenue_per_user_uplift_pct,
    CASE
        WHEN treatment_conversion_rate_pct > control_conversion_rate_pct
             AND treatment_revenue_per_user > control_revenue_per_user
        THEN 'Treatment Wins'
        WHEN treatment_conversion_rate_pct < control_conversion_rate_pct
             AND treatment_revenue_per_user < control_revenue_per_user
        THEN 'Control Wins'
        ELSE 'Mixed Result'
    END AS experiment_result
FROM pivoted
ORDER BY revenue_per_user_uplift_pct DESC;


CREATE OR REPLACE VIEW bi_experiment_significance_summary AS
WITH base AS (
    SELECT
        experiment_id,
        experiment_name,
        experiment_goal,
        variant,
        COUNT(*) AS users,
        SUM(converted) AS conversions,
        1.0 * SUM(converted) / COUNT(*) AS conversion_rate
    FROM experiments_clean
    GROUP BY experiment_id, experiment_name, experiment_goal, variant
),

pivoted AS (
    SELECT
        experiment_id,
        experiment_name,
        experiment_goal,
        MAX(CASE WHEN variant = 'control' THEN users END) AS control_users,
        MAX(CASE WHEN variant = 'treatment' THEN users END) AS treatment_users,
        MAX(CASE WHEN variant = 'control' THEN conversions END) AS control_conversions,
        MAX(CASE WHEN variant = 'treatment' THEN conversions END) AS treatment_conversions,
        MAX(CASE WHEN variant = 'control' THEN conversion_rate END) AS control_conversion_rate,
        MAX(CASE WHEN variant = 'treatment' THEN conversion_rate END) AS treatment_conversion_rate
    FROM base
    GROUP BY experiment_id, experiment_name, experiment_goal
),

stats AS (
    SELECT
        *,
        1.0 * (control_conversions + treatment_conversions) / (control_users + treatment_users) AS pooled_conversion_rate
    FROM pivoted
),

z_calc AS (
    SELECT
        *,
        SQRT(
            pooled_conversion_rate
            * (1 - pooled_conversion_rate)
            * (1.0 / control_users + 1.0 / treatment_users)
        ) AS standard_error
    FROM stats
)

SELECT
    experiment_id,
    experiment_name,
    experiment_goal,
    control_users,
    treatment_users,
    control_conversions,
    treatment_conversions,
    ROUND(100 * control_conversion_rate, 2) AS control_conversion_rate_pct,
    ROUND(100 * treatment_conversion_rate, 2) AS treatment_conversion_rate_pct,
    ROUND(100 * (treatment_conversion_rate - control_conversion_rate), 2) AS conversion_lift_pp,
    ROUND(100 * (treatment_conversion_rate - control_conversion_rate) / NULLIF(control_conversion_rate, 0), 2) AS conversion_uplift_pct,
    ROUND((treatment_conversion_rate - control_conversion_rate) / NULLIF(standard_error, 0), 2) AS z_score,
    CASE
        WHEN ABS((treatment_conversion_rate - control_conversion_rate) / NULLIF(standard_error, 0)) >= 1.96
        THEN 'Statistically Significant'
        ELSE 'Not Significant'
    END AS significance_result,
    CASE
        WHEN treatment_conversion_rate > control_conversion_rate
             AND ABS((treatment_conversion_rate - control_conversion_rate) / NULLIF(standard_error, 0)) >= 1.96
        THEN 'Roll Out Treatment'
        WHEN treatment_conversion_rate > control_conversion_rate
        THEN 'Promising, Need More Data'
        ELSE 'Do Not Roll Out'
    END AS recommendation
FROM z_calc
ORDER BY z_score DESC;
"""

# **Created 10_bi_ready_views.sql**

In [23]:
sql_files["10_bi_ready_views.sql"] = """
-- 10_bi_ready_views.sql
-- Purpose: List all final BI-ready views used in Power BI.

-- Final exported BI-ready views:
-- 1. bi_executive_summary_kpis
-- 2. bi_monthly_revenue_kpis
-- 3. bi_segment_churn_revenue
-- 4. bi_rfm_customer_segments
-- 5. bi_rfm_segment_summary
-- 6. bi_cohort_retention_summary
-- 7. bi_cohort_milestone_summary
-- 8. bi_recommendation_model_performance
-- 9. bi_recommendation_segment_performance
-- 10. bi_personalized_vs_generic_uplift
-- 11. bi_experiment_ab_summary
-- 12. bi_experiment_uplift_summary
-- 13. bi_experiment_significance_summary


-- Preview executive KPIs
SELECT * FROM bi_executive_summary_kpis;


-- Preview monthly revenue KPIs
SELECT * FROM bi_monthly_revenue_kpis ORDER BY month;


-- Preview segment churn and revenue
SELECT * FROM bi_segment_churn_revenue;


-- Preview RFM segment summary
SELECT * FROM bi_rfm_segment_summary ORDER BY total_monetary_value DESC;


-- Preview cohort milestone summary
SELECT * FROM bi_cohort_milestone_summary ORDER BY cohort_month;


-- Preview recommendation model performance
SELECT * FROM bi_recommendation_model_performance ORDER BY recommendation_revenue DESC;


-- Preview personalized vs generic uplift
SELECT * FROM bi_personalized_vs_generic_uplift ORDER BY revenue_per_impression_uplift_pct DESC;


-- Preview experiment uplift summary
SELECT * FROM bi_experiment_uplift_summary ORDER BY revenue_per_user_uplift_pct DESC;


-- Preview experiment significance summary
SELECT * FROM bi_experiment_significance_summary ORDER BY z_score DESC;
"""

# **Save all SQL Views in zip file** 

In [24]:
# Write all SQL files
for filename, content in sql_files.items():
    file_path = os.path.join(sql_folder, filename)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content.strip() + "\n")
    print(f"Created: {file_path}")

# Zip SQL folder
zip_path = "/kaggle/working/sql_files"
shutil.make_archive(zip_path, "zip", sql_folder)

print("\nAll SQL files created successfully.")
print("Download ZIP:", zip_path + ".zip")

Created: /kaggle/working/sql/01_data_validation.sql
Created: /kaggle/working/sql/02_clean_views.sql
Created: /kaggle/working/sql/03_executive_kpis.sql
Created: /kaggle/working/sql/04_churn_analysis.sql
Created: /kaggle/working/sql/05_revenue_analysis.sql
Created: /kaggle/working/sql/06_rfm_segmentation.sql
Created: /kaggle/working/sql/07_cohort_analysis.sql
Created: /kaggle/working/sql/08_personalization_analysis.sql
Created: /kaggle/working/sql/10_bi_ready_views.sql
Created: /kaggle/working/sql/09_ab_testing_analysis.sql

All SQL files created successfully.
Download ZIP: /kaggle/working/sql_files.zip
